# Information retrieval with BERT-family encoders and LLMs

This notebook builds, runs and evaluates the retrieval pipeline behind most "search + LLM" systems:

```
query ──► BM25 (lexical) ─────────┐
      └─► bi-encoder (dense) ─────┴─► rank fusion (RRF) ─► LLM answer with cited sources
```

Every stage is measured on **real documents, real queries and real relevance judgments (qrels)** from two public benchmark collections:

| Collection | Used in | Documents | Test queries | Typical query | Judged relevant per query |
|---|---|---|---|---|---|
| **SciFact** (BEIR version) | Part A | 5,183 scientific abstracts | 300 | a one-sentence scientific claim (~12 words) | ~1.1 (binary) |
| **NFCorpus** (BEIR version) | Part B | 3,633 medical abstracts | 323 | a topic word or a consumer-health title (~2 words) | ~38 (graded 1–2) |

**Part A: the pipeline (SciFact).** 1 Setup · 2 Data and qrels · 3 Evaluation measures · 4 BM25 · 5 Dense retrieval · 6 Hybrid retrieval ·
7 An LLM answering from retrieved passages · 8 Summary of results.

**Part B: further experiments.** Each one needs only Sections 1–2 and can be run on its own.

| Section | Content |
|---|---|
| B1 | The same pipeline on NFCorpus (short queries, graded relevance) |
| B2 | SciFact queries grouped by measurable properties, with many examples and negation pairs |
| B3 | Controlled query reformulations (paraphrase, synonym, keywords, long question, spelling) |
| B4 | Encoder choice and similarity function (5 encoders, dev and test) |
| B5 | BM25 preprocessing and k1/b, tuned on dev, evaluated once on test |
| B6 | Fusion settings: RRF k, fusion depth, score-based fusion |
| B7 | LLM: number of sources given to the model |
| B8 | Paired bootstrap intervals for the main comparisons |

Results are stored by their parameters (see "How results are cached" in Section 1), so re-running a cell, or running a Part B section
directly after Sections 1–2, gives the same result regardless of the order in which other cells were run.

## Key terms

* **Encoder.** A Transformer that reads a whole token sequence and outputs one contextual vector per token (BERT, DistilBERT,
  MiniLM). It does not generate text. To get *one* vector for a whole text, the token vectors must be **pooled**, for example by taking
  the `[CLS]` vector or the mean of all token vectors.
* **Bi-encoder (dense retriever).** The query and the document are encoded **separately**, each into a single vector. Relevance is a
  similarity of the two vectors (cosine or dot product). Because documents do not depend on the query, they are encoded once, before
  any query arrives. At query time only the query is encoded and compared with the stored vectors, which is fast and scales to millions
  of documents with an index. The price is that query and document never "see" each other inside the model.
* **Cross-encoder.** For contrast: a model that reads query and document **together** (`[CLS] query [SEP] document [SEP]`) and outputs
  one relevance score. It can relate every query token to every document token, but nothing can be pre-computed, so it is only usable
  on short candidate lists. It is not used in this notebook.
* **Generative LLM (decoder).** A Transformer trained to predict the next token. Given a prompt that includes retrieved passages, it
  writes an answer token by token. It has no built-in notion of which passage is true or relevant. Its citations are generated text like
  any other, so they must be checked.
* **BM25.** A lexical ranking function. A document scores high if it contains the query's terms, especially rare terms, with
  diminishing returns for repeated terms (`k1`) and a correction for document length (`b`). No training is involved.

**Why a general pretrained BERT is not automatically a good retriever.** `bert-base-uncased` was trained to fill in masked words
and to predict whether two sentences are consecutive. Neither objective teaches it that a *claim* and the *abstract that addresses it*
should map to nearby vectors. Its `[CLS]` vector was never trained as a text representation for similarity. Mean-pooled token vectors
from such a model are dominated by frequent, generic directions, so many unrelated texts look similar (Ethayarajh, 2019). Retrieval
models start from a BERT-style encoder but are then **trained on query–document pairs** (e.g. MS MARCO search queries and the passages
users clicked) with a contrastive objective that pulls relevant pairs together and pushes irrelevant ones apart. Section 5 measures the
difference on the same collection.

## 1 · Setup and configuration

The next cell imports the helpers (the `irlab/` package in this repository), defines **all experiment settings in one `LabConfig`
object**, fixes the random seeds, chooses the device and prints the software versions.

* *Settings:* BM25 uses the textbook/Elasticsearch defaults `k1 = 1.2, b = 0.75`, fixed **before** looking at any results. The dense
  encoder `bge-small` was selected on **development** queries (Section B4). RRF uses `k = 60` from the original paper (Cormack et al., 2009).
  Each choice is examined in Part B.
* *Device:* `"auto"` means the GPU if PyTorch can see one, otherwise the CPU. It is set in the `.env` file of the project
  (`IRLAB_DEVICE=auto`, `cpu` or `cuda`). The device actually used is printed as `device_used`.
* *Where it can fail:* no internet on first use (models and data come from the Hugging Face Hub), or a GPU that runs out of memory
  (lower `dense_batch_size`).

### How results are cached

Some steps take minutes (encoding thousands of abstracts, downloading a 3 GB language model) and others take milliseconds (scoring a query).
The notebook therefore stores the expensive results and recomputes only the cheap ones. There are three layers, each explained again where it is
first used:

| Layer | Where | What is stored | When it is reused | When it is recomputed |
|---|---|---|---|---|
| Downloads | `data/beir/<collection>/`, `data/scifact_release/`, `~/.cache/huggingface/` | the collection files at a pinned commit, the original SciFact release, model weights | whenever the files are already present | never, unless the folder is deleted |
| Corpus embeddings | `cache/embeddings/*.npy` + `.json` | one vector per document, plus metadata (model, device, encoding time) | when the model, its revision, the maximum length **and** the corpus text are all identical | as soon as any of these four changes, because the file name changes |
| In-memory results | the `lab` object (Python dictionaries) | every BM25 and dense run, keyed by all of its parameters; BM25 indexes; loaded models (RRF runs are recomputed from the stored BM25 and dense runs, which takes milliseconds) | for the rest of the session, when a cell asks for a run with the same parameters | after a kernel restart (in-memory results are never written to disk) |

What is **not** cached: query embeddings (one forward pass per query, milliseconds), the evaluation measures (computed from the runs every time),
and LLM answers (generated afresh each run; with greedy decoding they come out the same on the same hardware and software).

The three folders are set in the project's `.env` file (`IRLAB_DATA_DIR`, `IRLAB_CACHE_DIR`, `IRLAB_RESULTS_DIR`), which `irlab` reads when it is imported. Deleting
`cache/` is always safe: the next run recomputes the embeddings and stores them again.

### The main objects of this notebook

Every code cell uses the same few objects. They are defined in the `irlab/` package and explained here once. The cells before later code
cells repeat only the parts that cell uses.

**Data structures (plain Python dictionaries).**

| name | type | example | meaning |
|---|---|---|---|
| queries | `dict[str, str]` | `{"129": "Articles published in open access format are less likely to be cited ..."}` | query id → query text |
| qrels | `dict[str, dict[str, int]]` | `{"129": {"27768226": 1}}` | query id → {document id → relevance label}. A document that is not listed is *unjudged*. |
| **run** | `dict[str, list[tuple[str, float]]]` | `{"129": [("27768226", 23.4), ("4414547", 18.9), ...]}` | the output of one retrieval system for a set of queries: for every query id, the retrieved document ids with their scores, **best first**, at most `retrieval_depth` (100) of them. Every retrieval method in this notebook (BM25, dense, RRF) returns a run, and every evaluation function takes runs. |

A run is the standard unit of IR evaluation (TREC calls the same thing a "run file"): the system is run once over all queries, and the
resulting ranked lists are compared with the qrels. Two runs can be compared only if they cover the same queries.

**`LabConfig`: all settings in one object.** A frozen dataclass (`irlab/config.py`). Frozen means its fields cannot be changed after it is
created, so the settings printed below are guaranteed to be the settings used. To try another value, a cell passes it as a parameter to a `lab`
method (for example `rrf_k=10`), which leaves `CFG` itself unchanged. The fields:

| field | value here | what it controls |
|---|---|---|
| `seed` | 13 | seed for Python, NumPy and PyTorch random generators, and for the bootstrap resampling of B8 |
| `device` | `"auto"` | where models run: `"auto"` = GPU if PyTorch can see one, else CPU; or `"cpu"` / `"cuda"`. Set by `IRLAB_DEVICE` in `.env` |
| `data_dir`, `cache_dir`, `results_dir` | `data/`, `cache/`, `results/` | folders for downloads, embedding files and saved tables (`IRLAB_DATA_DIR`, `IRLAB_CACHE_DIR`, `IRLAB_RESULTS_DIR` in `.env`; relative paths are relative to the project folder) |
| `retrieval_depth` | 100 | how many documents every BM25, dense and RRF run keeps per query; R@100 is measured on these |
| `bm25_k1`, `bm25_b` | 1.2, 0.75 | BM25 term-frequency saturation and length normalisation (Section 4) |
| `bm25_stopwords`, `bm25_stem` | True, True | BM25 preprocessing: remove 33 English function words; apply the Snowball stemmer |
| `dense_model` | `"bge-small"` | key of the default encoder in `CFG.encoders` (the five encoders are listed in B4) |
| `dense_batch_size` | 64 | texts encoded together in one forward pass; lower it if the GPU runs out of memory. It changes speed, not results |
| `rrf_k` | 60 | the constant *k* of reciprocal rank fusion (Section 6) |
| `llm_id`, `llm_revision` | Qwen2.5-1.5B-Instruct, pinned commit | the generative model of Section 7 |
| `llm_max_new_tokens` | 220 | upper limit on the length of a generated answer |
| `llm_n_sources` | 3 | how many retrieved passages are put into the prompt |
| `llm_passage_words` | 300 | each passage is cut to its first 300 words |
| `eval_cutoffs` | (10, 100) | the *k* values of the measures |
| `bootstrap_samples` | 10,000 | number of resamples in B8 |
| `collections`, `encoders` | registries | where each collection is downloaded from (repository and commit) and how each encoder is called (model id, commit, similarity, prompts). Defined in `irlab/config.py` |

`.env` holds only these machine-specific values (device and folders). `irlab` loads it with `python-dotenv` when it is imported, so it must be edited before the kernel starts, or the kernel restarted afterwards. A variable set in the shell takes precedence over the file.

`CFG.summary()` returns the fields that affect results, without paths and registries, for printing.

**`Lab`: builds runs on request and remembers them.** `lab = Lab(CFG)` fixes the seeds, resolves `"auto"` to an actual device (stored in
`lab.device`) and creates the cache and results folders. Afterwards, every cell asks `lab` for what it needs:

| method | returns | main parameters |
|---|---|---|
| `lab.collection(name)` | a `Collection` (below) | `name`: `"scifact"` or `"nfcorpus"`. Downloaded on first use, then kept in memory |
| `lab.split_name(name, role)` | the BEIR split name | `role`: `"dev"` or `"test"`. For SciFact, dev is BEIR's `train` split |
| `lab.queries(name, split)` | queries | the queries that have judgments in that split |
| `lab.qrels(name, split)` | qrels | the judgments of that split |
| `lab.bm25_run(name, queries, prep=None, k1=None, b=None, depth=None)` | run | `prep`: a `Preprocessing` (stopwords on/off, stemming on/off) |
| `lab.dense_run(name, queries, model_key=None, similarity=None, max_seq_length=None, depth=None)` | run | `model_key`: an encoder key; `similarity`: `"cos"` or `"dot"` |
| `lab.hybrid_run(name, queries, rrf_k=None, fusion_depth=None)` | run | RRF of the BM25 run and the dense run of the same queries |
| `lab.evaluate(name, split, runs)` | table, one row per run | `runs`: `{label: run}`; the label becomes the `system` column |
| `lab.per_query(name, split, run)` | table, one row per query | the same measures, per query (used for examples and the bootstrap) |
| `lab.timing_table()` | table | every timed step of the session (setup, query, generate) |
| `lab.save_table(df, filename)` | nothing | writes `df` as CSV to `results/` and prints the path |
| `lab.environment()` | dict | software versions, GPU name, device used |

A parameter left as `None` takes its value from `CFG`. Every run is stored inside `lab` under a key made of **all** its parameters (collection,
method, model, similarity, preprocessing, k1, b, depth, and a hash of the query texts). Asking again for the same run returns the stored one
instead of recomputing it, and asking with any parameter changed computes a new one. This is the third cache layer of the table above.

**`Collection`** (`irlab/data.py`), returned by `lab.collection(...)`: `doc_ids` (list of document ids, in corpus order), `titles`, `texts`,
`queries` (all queries of the collection), `qrels` (`{split: qrels}`), `doc_index` (document id → position in the corpus, used to find a
document's row in a matrix), `full_text(i)` (title + abstract of the document at position `i`), and `fingerprint()` (a hash of all ids and texts).

**What the next cell does.** It imports these objects and the helper modules (`metrics`, `analysis`, `generation`, `display` imported as
`show`), creates `CFG` and `lab`, and prints the settings and the software environment.

In [1]:
# Configuration and setup: the only place where experiment settings are chosen.
import os
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import inspect, json, re, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
import transformers

from irlab import Lab, LabConfig, Preprocessing
from irlab import analysis, generation, metrics
from irlab import display as show
from irlab.data import describe, load_scifact_evidence
from irlab.lexical import tokenize

transformers.logging.set_verbosity_error()
transformers.utils.logging.disable_progress_bar()
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 100)

CFG = LabConfig(
    seed=13,
    # "auto" = GPU if PyTorch sees one, else CPU; or "cpu" / "cuda". Read from IRLAB_DEVICE in the project's .env file.
    device=os.environ.get("IRLAB_DEVICE", "auto"),
    retrieval_depth=100,            # every first-stage run keeps its top 100 documents
    bm25_k1=1.2, bm25_b=0.75,       # a-priori defaults (tuned only in B5, on dev data)
    bm25_stopwords=True, bm25_stem=True,
    dense_model="bge-small",        # chosen on dev data, see B4
    rrf_k=60,                       # Cormack et al. 2009
    llm_n_sources=3,                # studied in B7
)
lab = Lab(CFG)
print("settings:", json.dumps(CFG.summary(), default=str))
print("environment:", json.dumps(lab.environment(), indent=1))

settings: {"seed": 13, "device": "auto", "retrieval_depth": 100, "bm25_k1": 1.2, "bm25_b": 0.75, "bm25_stopwords": true, "bm25_stem": true, "dense_model": "bge-small", "dense_batch_size": 64, "rrf_k": 60, "llm_id": "Qwen/Qwen2.5-1.5B-Instruct", "llm_revision": "989aa7980e4cf806f80c7fef2b1adb7bc71aa306", "llm_max_new_tokens": 220, "llm_n_sources": 3, "llm_passage_words": 300, "eval_cutoffs": [10, 100], "bootstrap_samples": 10000}


environment: {
 "python": "3.13.12",
 "platform": "Linux-7.0.0-43-generic-x86_64-with-glibc2.39",
 "torch": "2.15.0+cu132",
 "cuda": "13.2",
 "gpu": "NVIDIA GeForce RTX 5070 Ti",
 "device_used": "cuda",
 "sentence_transformers": "6.3.0",
 "transformers": "5.24.1",
 "bm25s": "0.3.16",
 "numpy": "2.5.5",
 "pandas": "3.1.1"
}


The printed versions belong next to any reported number. Results can shift slightly across library versions and between CPU and GPU
(the floating-point summation order differs).

## 2 · The collection and its relevance judgments

**SciFact** (Wadden et al., EMNLP 2020) pairs expert-written scientific **claims** with the research **abstracts** that address them.
We use the [BEIR](https://github.com/beir-cellar/beir) version (Thakur et al., NeurIPS 2021 Datasets & Benchmarks), in the standard
*corpus / queries / qrels* format:

* `corpus`: 5,183 abstracts (title + text), drawn from S2ORC;
* `queries`: 1,109 claims;
* `qrels`: `(query id, document id, label)` triples. Every label is 1, and documents without a qrels entry are **unjudged**.

**Splits.** BEIR's SciFact `test` split (300 claims) is the original SciFact **dev** set, because the original test labels were never
released. BEIR's `train` split (809 claims) is the original training set. Nothing is trained on it here. It serves as the
**development set** for choosing settings, and chosen settings are evaluated once on `test`.

**What "relevant" means here** (checked against the original SciFact release). BEIR's SciFact qrels contain *every abstract cited for
the claim* (339 pairs for 300 test claims), including abstracts in which the SciFact annotators found **no** rationale. Only 209 of those
339 pairs have a SUPPORT or CONTRADICT rationale, and many claims were deliberately written as **false** (negated) versions of a finding.
So a "relevant" document is *about* the claim; it may contradict it or not settle it at all.

**Why SciFact.** It is small enough to encode on a laptop CPU. Its claims are full sentences with technical terms, gene names and
abbreviations, exactly where lexical and dense retrieval behave differently. Its licences (claims CC BY 4.0, abstracts ODC-By 1.0) allow
teaching use. Its official rationale labels make it possible to check LLM answers *separately* from retrieval.

**What the next cell does, step by step.**

1. `lab.collection("scifact")` calls `huggingface_hub.snapshot_download` for the corpus repository (`BeIR/scifact`) and the qrels repository
   (`BeIR/scifact-qrels`), each at a fixed commit hash, into `data/beir/scifact/`. If the files are already there, nothing is downloaded. This is
   the first cache layer.
2. It reads the corpus and queries (Parquet) and the qrels (TSV) into Python structures. Each document becomes `title + " " + abstract`.
3. It checks the number of documents, queries and judgments per split against the BEIR documentation and **stops with an error** if anything
   differs, so a truncated or changed download cannot silently produce different numbers. It also checks that no query is in both dev and test.
4. The collection object is kept in memory, so later cells reuse it without re-reading the files.
5. `describe()` prints one row per split.

**Code in the next cell.**

* `C = "scifact"`: the collection name used in every Part A cell.
* `lab.split_name(C, "dev")` → `"train"`, `lab.split_name(C, "test")` → `"test"`, stored as `DEV` and `TEST`.
* `lab.collection(C)` → the `Collection` object `sci` (downloads, reads and checks the files, as described above).
* `lab.queries(C, TEST)` and `lab.qrels(C, TEST)` → `test_q` (300 claims) and `test_qrels`; the same for dev.
* `describe(collection)` (`irlab/data.py`) → a table with one row per split: number of queries and judgments, judged documents per query,
  label counts, median query and document length.

In [2]:
# Load SciFact and describe its splits.
C = "scifact"
DEV, TEST = lab.split_name(C, "dev"), lab.split_name(C, "test")
sci = lab.collection(C)                      # raises ValueError if sizes differ from the BEIR documentation
test_q, test_qrels = lab.queries(C, TEST), lab.qrels(C, TEST)
dev_q, dev_qrels = lab.queries(C, DEV), lab.qrels(C, DEV)
display(describe(sci))

,collection,split,role,queries,judgments,judged/query (mean),judged/query (median),label counts,query words (median),docs,doc words (median)
0,scifact,test,test (final),300,339,1.13,1.0,{1: 339},12.0,5183,204.0
1,scifact,train,dev (tuning),809,919,1.14,1.0,{1: 919},12.0,5183,204.0


The `test` row has 300 queries and 339 judgments, all with label 1: most claims have exactly **one** judged-relevant abstract. Claims have a
median of 12 words, and abstracts about 200. This has two consequences for evaluation. (1) MRR@10 and nDCG@10 will be close to each
other. (2) An unjudged document in the top ranks is *counted as non-relevant* even if a person would find it useful. The scores are lower
bounds on usefulness, and they stay comparable across systems only if no system is systematically favoured by the judging process.

The next cell shows claim 129, its judged abstract, and the label it has in the original SciFact release. That label comes from the original
SciFact data (`claims_dev.jsonl` inside `data.tar.gz`). It is downloaded once to `data/scifact_release/`, verified by its sha256 checksum, and
used only to check LLM answers in Section 7, never to evaluate retrieval.

**Code in the next cell.**

* `load_scifact_evidence(data_dir)` → `{claim id: {"claim": text, "evidence": {doc id: "SUPPORT" | "CONTRADICT"}, "cited": [doc ids]}}`
  from the original SciFact release. A cited document without an entry in `evidence` has no rationale.
* `show.show_judged(collection, split, qid, words=110, evidence=None)` prints the query, then every judged document with its qrels label
  (and its SciFact label if `evidence` is given) and the first `words` words of its text.

In [3]:
# One claim, its judged abstract, and the original SciFact rationale label.
evidence = load_scifact_evidence(CFG.data_dir)   # original SciFact dev labels; used only to check LLM answers
show.show_judged(sci, TEST, "129", evidence=evidence)

QUERY 129: Articles published in open access format are less likely to be cited than traditional journals.

[27768226] qrels label 1, SciFact rationale label: CONTRADICT
Open Access Increases Citation Rate PLoS Biology publishes today a research article by Gunther
Eysenbach that is not about biology. It is about citations. It provides robust evidence that open-
access articles (OA articles) are more immediately recognized and cited than non-OA articles. As
such, it adds objective support to the belief we have always held that open-access publication
speeds up scientific dialog between researchers and, consequently, should be extended to the whole
scientific literature as quickly as possible. It is therefore fitting that we publish such a paper.
We have long argued that papers freely available in a journal will be more often read and cited than
those behind a ...


The abstract is judged relevant (label 1) because it is the cited source for the claim. Its title, however, states the opposite of the
claim (*Open Access Increases Citation Rate*), and SciFact labels it CONTRADICT. A retrieval system that ranks this abstract first has
done its job. A generator that then concludes "open-access articles are cited less" has not. **Retrieval metrics measure whether the
right documents were found, not whether an answer built from them is correct.**

## 3 · Evaluation measures

For one query, let the system return a ranked list and let `rel(d)` be the qrels label of document `d` (0 if unjudged).

* **Recall@k** = (number of relevant documents in the top k) / (number of relevant documents). It answers *were they found at all?*
  **R@100** describes the candidate pool of a first stage, which is what fusion and the LLM step draw from.
* **MRR@k** = mean over queries of 1/(rank of the first relevant document), or 0 if none is in the top k. It answers *how far must a
  user scroll to the first hit?* It ignores all relevant documents after the first.
* **nDCG@k** = DCG@k / IDCG@k with DCG@k = Σ_{i≤k} rel(dᵢ) / log₂(i + 1) and IDCG the DCG of the ideal ordering of *all* judged
  documents. It rewards putting relevant (and, if graded, more relevant) documents near the top. **nDCG@10 is BEIR's main measure.**

Every query in the qrels is averaged, and a query with nothing retrieved scores 0. The implementation (`irlab/metrics.py`) follows
trec_eval conventions and is checked against [`ir_measures`](https://ir-measur.es/) in the next cell, on a
small example: a query with two judged documents, `dA` (label 2) and `dB` (label 1), and a system that returns `dX, dA, dY, dB, dZ`.

**Code in the next cell.**

* `metrics.recall_at_k(ranked, rels, k)`, `metrics.rr_at_k(ranked, rels, k)`, `metrics.ndcg_at_k(ranked, rels, k)`: the three measures for
  **one** query. `ranked` is a list of document ids, best first; `rels` is that query's qrels entry, `{doc id: label}`.
* `inspect.getsource(...)` prints the source code of `ndcg_at_k`, so the implementation can be compared with the formula.
* `ir_measures.calc_aggregate(measures, qrels, run)`: the reference library. It takes the qrels as `Qrel(query, doc, label)` records and the run
  as `ScoredDoc(query, doc, score)` records.

For a whole run, `metrics.evaluate(run, qrels, measures)` averages each measure over all judged queries, and `lab.evaluate` calls it once
per run.

In [4]:
# Measures on a small example, and a cross-check against ir_measures (trec_eval conventions).
toy_rels = {"dA": 2, "dB": 1}
toy_ranking = ["dX", "dA", "dY", "dB", "dZ"]
print("R@2     =", metrics.recall_at_k(toy_ranking, toy_rels, 2))
print("MRR@10  =", metrics.rr_at_k(toy_ranking, toy_rels, 10))
print("nDCG@10 =", round(metrics.ndcg_at_k(toy_ranking, toy_rels, 10), 4))
print(inspect.getsource(metrics.ndcg_at_k))

import ir_measures
from ir_measures import Qrel, ScoredDoc
run_toy = {"q": [(d, float(len(toy_ranking) - i)) for i, d in enumerate(toy_ranking)]}
ref = ir_measures.calc_aggregate([ir_measures.nDCG @ 10, ir_measures.RR @ 10],
                                 [Qrel("q", d, r) for d, r in toy_rels.items()],
                                 [ScoredDoc("q", d, s) for d, s in run_toy["q"]])
print("ir_measures:", {str(k): round(v, 4) for k, v in ref.items()})

R@2     = 0.5
MRR@10  = 0.5
nDCG@10 = 0.6433
def ndcg_at_k(ranked: list[str], rels: dict[str, int], k: int) -> float:
    dcg = sum(rels.get(d, 0) / math.log2(pos + 1) for pos, d in enumerate(ranked[:k], start=1))
    ideal = sorted((r for r in rels.values() if r > 0), reverse=True)[:k]
    idcg = sum(g / math.log2(pos + 1) for pos, g in enumerate(ideal, start=1))
    return dcg / idcg if idcg > 0 else 0.0

ir_measures: {'nDCG@10': 0.6433, 'RR@10': 0.5}


The values follow directly from the definitions. R@2 = 1/2, because only `dA` is in the top 2. MRR@10 = 1/2, because the first relevant
document is at rank 2. DCG = 2/log₂3 + 1/log₂5 = 1.262 + 0.431 = 1.692 and IDCG = 2/log₂2 + 1/log₂3 = 2.631, so nDCG@10 = 0.643. `ir_measures`
gives the same values. `dX`, `dY` and `dZ` are unjudged and are treated as non-relevant.

## 4 · Lexical retrieval with BM25

For a query with terms t and a document d,

  BM25(q, d) = Σ_t IDF(t) · tf(t,d)·(k1 + 1) / ( tf(t,d) + k1·(1 − b + b·|d|/avgdl) ).

* `IDF(t)` is large for rare terms, so matching a gene name counts much more than matching "cells".
* `k1` controls term-frequency saturation: with k1 → 0 only presence matters; larger k1 lets repeated terms keep adding.
* `b` controls length normalisation: b = 0 ignores length, b = 1 fully normalises by length relative to the average.

**Preprocessing decides what a "term" is.** The tokenizer lowercases, splits on anything that is not `a–z` or `0–9`, removes 33 English
function words and applies the Snowball English stemmer (so *cited/citation/citations* → `cite`/`citat`…). Section B5 measures this
choice. Two weaknesses matter for biomedical text: `IL-6` becomes `il` + `6`, and Greek letters such as `α` are **dropped**.

**What the next cell does, step by step.**

1. **Indexing (setup).** Every document (title + abstract) is tokenized as described above. `bm25s` then builds a sparse matrix that stores, for
   every term and every document containing it, the term's BM25 weight (IDF × saturated, length-normalised term frequency). This takes about
   half a second for 5,183 abstracts. The index lives **in memory only**: it is cheap to rebuild, so it is not written to disk. The `lab` object
   keeps it for the session, keyed by (collection, preprocessing, k1, b).
2. **Searching (query time).** Each of the 300 claims is tokenized the same way. Terms that do not occur in the index are dropped. The score of every
   document is the sum of the stored weights of the query's terms. Documents are sorted by score (ties broken by document position, so the
   result is deterministic), and the top 100 with a score above 0 are kept. A document that shares no term with the query is not retrieved at
   all, as in Lucene. A query whose every term is unknown returns an empty list and scores 0.
3. **Evaluation.** The run is compared with the test qrels using the measures of Section 3.

The result is a *run*, a dictionary `{claim id: [(document id, score), …]}` sorted best-first, stored in the `lab` object under all of its
parameters.

**Code in the next cell.**

* `lab.bm25_run(C, test_q)` → a run. With no other arguments it uses `CFG`: stopwords and stemming on, `k1 = 1.2`, `b = 0.75`, depth 100.
  Internally `lab.bm25_index(C)` builds (or reuses) a `BM25Index` for that preprocessing and those parameters, and `index.search(text, depth)`
  returns the positions and scores of the best documents for each query.
* `lab.evaluate(C, TEST, {label: run})` → one row with nDCG@10, MRR@10, R@10 and R@100 averaged over the 300 test claims. It raises an error if
  the run lacks any judged query, so a run cannot be evaluated on a subset by accident.
* `lab.timing_table()` → the timed steps so far; the cell keeps the rows whose component contains "BM25".

In [5]:
# BM25 with a-priori parameters on the SciFact test claims.
bm25_test = lab.bm25_run(C, test_q)
display(lab.evaluate(C, TEST, {"BM25 (stop+stem, k1=1.2, b=0.75)": bm25_test}))
display(lab.timing_table().query("component.str.contains('BM25')", engine="python"))

,collection,split,system,queries,nDCG@10,MRR@10,R@10,R@100
0,scifact,test,"BM25 (stop+stem, k1=1.2, b=0.75)",300,0.6862,0.6506,0.8168,0.9209


,stage,component,collection,seconds,items,ms/item,device,from_cache,note
1,setup,"BM25 index (stop+stem, k1=1.2, b=0.75)",scifact,0.460,5183.0,0.09,cuda,False,
2,query,BM25 search,scifact,0.052,300.0,0.17,cuda,False,


This is BM25's effectiveness on the complete BEIR SciFact test split, all 300 claims against all 5,183 abstracts. For reference, the BEIR paper (Table 2) reports nDCG@10 = 0.665 for its
Elasticsearch BM25. Differences of a couple of points come from tokenization, stemming and the BM25 variant. The second table separates
**setup** (building the index, done once) from **query** time (all 300 queries together). Both take well under a second at this corpus size.

The next cell shows what BM25 actually matches: the index terms of two claims with their document frequency (df), followed by the BM25 top 5.

**Code in the next cell.**

* `lab.bm25_index(C)` → the `BM25Index` already built by the previous cell (from memory). Its attributes used here: `prep` (the
  `Preprocessing` it was built with) and `doc_freq` (`{term: number of documents containing it}`).
* `tokenize(text, prep)` (`irlab/lexical.py`) → the list of index terms of a text: lowercased, split on non-`[a-z0-9]` characters, stopwords
  removed and stemmed according to `prep`. It is the same function used for indexing, so the printed terms are exactly what BM25 matches.
* `show.compare_systems(collection, runs, qid, query_text, rels, k=5)` prints the query, its judged-relevant documents, and the top `k` of
  each run in `runs` (`{label: run}`). Each line shows rank, document id, qrels label (or `unjudged`), score and the start of the text; relevant
  documents are marked with `*`. The header of each list gives the rank of the first relevant document.

In [6]:
# What BM25 sees: index terms and their document frequency (df), then the BM25 top 5.
idx = lab.bm25_index(C)
for qid in ["248", "660"]:
    terms = tokenize(test_q[qid], idx.prep)
    print(f"{qid}: {test_q[qid]}")
    print("   terms (df):", ", ".join(f"{t}({idx.doc_freq.get(t, 0)})" for t in terms))
for qid in ["248", "660"]:
    show.compare_systems(sci, {"BM25": bm25_test}, qid, test_q[qid], test_qrels[qid], k=5)

248: Chenodeosycholic acid treatment increases whole-body energy expenditure.
   terms (df): chenodeosychol(0), acid(405), treatment(911), increas(1573), whole(122), bodi(313), energi(146), expenditur(29)
660: Ivermectin is used to treat onchocerciasis.
   terms (df): ivermectin(0), use(2182), treat(405), onchocerciasi(1)
QUERY 248: Chenodeosycholic acid treatment increases whole-body energy expenditure.
judged relevant (1): 1568684 (label 1)
  [1568684] The Bile Acid Chenodeoxycholic Acid Increases Human Brown Adipose Tissue Activity. The
        interest in brown adipose tissue (BAT) as a target to combat metabolic disease has recently
        been renewed with the discovery of functional BAT in humans. In rodents, BAT can be
        activated by bile ...
--- BM25: first relevant at rank 1
 * 1. [1568684] (1)   13.815  The Bile Acid Chenodeoxycholic Acid Increases Human Brown Adipose Tissue Activity. The interest in brown adipose tissue (BAT) ...
   2. [25050364] (unjudged)   11.933 

* Claim **248** misspells *chenodeoxycholic* as **"Chenodeosycholic"**. The stem `chenodeosychol` has df = 0, so the word naming the
  substance contributes **nothing**. BM25 still ranks the relevant abstract (1568684) first, because the remaining terms, in particular
  the fairly rare `expenditur` (df = 29) together with `acid` and `energi`, all occur in it. A misspelling hurts only when the misspelled
  word carries most of the evidence (Section B3 runs the corrected spelling, and the rank stays 1).
* Claim **660**: `ivermectin` has df = 0. No abstract contains the word, not even the relevant one, which is about integrated control of
  neglected tropical diseases. But `onchocerciasi` has df = **1**: it occurs only in the relevant abstract, and its very high IDF alone puts
  that abstract at rank 1. Rare exact terms are BM25's strength.

## 5 · Dense retrieval with BERT-family bi-encoders

The next cell encodes all 5,183 abstracts with two encoders and searches the 300 test claims:

1. **`bert-base-uncased` + mean pooling.** A general pretrained BERT with no retrieval training, i.e. the "just use BERT" baseline.
2. **`BAAI/bge-small-en-v1.5`.** A 33M-parameter BERT-architecture encoder trained contrastively on large collections of text pairs for
   retrieval. It expects the query to be prefixed with an instruction (`"Represent this sentence for searching relevant passages: "`, set in
   `irlab/config.py`) and uses cosine similarity. It was selected among five encoders on the **dev** queries of both collections (B4). Its
   training data is very broad, and it is not documented whether it excludes PubMed-like abstracts, so its scores are "zero-shot with respect
   to these qrels", not proof of domain transfer.

**What happens inside, step by step.**

1. **Tokenization.** Each text is split into WordPiece sub-words (e.g. *Silencing* → `si`, `##len`, `##cing`), and `[CLS]` and `[SEP]` are added.
   At most **512 tokens** are read and the rest is silently dropped. That affects 455 of the 5,183 abstracts (about 9%) with this tokenizer.
2. **Encoding.** The Transformer turns the tokens into one contextual vector per token (768 numbers for bert-base, 384 for bge-small).
3. **Pooling.** For plain BERT we take the **mean** of the token vectors. bge-small was trained to use the **`[CLS]` vector**, which is then
   **normalised** to length 1.
4. **Query side.** Each claim goes through the same steps, but bge-small first prepends its instruction to the claim.
5. **Similarity and ranking.** The 300 query vectors are multiplied with the 5,183 document vectors, giving a 300 × 5,183 matrix of cosine
   similarities (the dot product of length-1 vectors is their cosine). For each claim, the 100 highest values are kept.

**Caching of the document vectors (the second cache layer).** Step 1–3 for 5,183 abstracts is the expensive part: seconds on a GPU, minutes on a
CPU. The resulting matrix is therefore saved to `cache/embeddings/` as a NumPy file, named by four things: model key, model revision, maximum length,
and a hash of the corpus (all document ids and texts, in order). Before encoding, the code looks for a file with exactly that name. If it exists, the
vectors are loaded in milliseconds. If not, they are computed and saved. Changing the model, its version, the length limit, or a single character
of any document produces a different name, and therefore fresh vectors, so stale vectors can never be used by mistake. Query vectors are *not*
cached: 300 short claims take well under a second. The cell after the results lists the cache files and one metadata record.

**Code in the next cell.**

* `lab.dense_run(C, test_q)` → a run with the default encoder `CFG.dense_model = "bge-small"` and the similarity it was trained with (cosine).
* `lab.dense_run(C, test_q, model_key="bert-base-mean")` → the same with plain BERT. Other parameters: `similarity` (`"cos"` or `"dot"`,
  default: the model's own), `max_seq_length` (default 512), `depth` (default 100).

Inside `dense_run`: `lab.doc_embeddings(C, model_key)` returns the document matrix (from `cache/embeddings/` if a matching file exists,
otherwise it encodes and saves it); the queries are encoded with `dense.encode_texts(spec, texts, role="query", ...)`, which adds the model's
query prompt; `dense.similarity_matrix(q, d, similarity, device)` multiplies the two matrices; `dense.top_k(scores, depth)` keeps the best
100 per query. Each encoder is described by an `EncoderSpec` in `CFG.encoders` (model id, pinned revision, similarity, query and document
prompts, maximum length).

In [7]:
# Plain BERT vs. a retrieval-trained BERT-family encoder, same queries and qrels.
dense_test = lab.dense_run(C, test_q)                                  # bge-small, cosine
bert_test = lab.dense_run(C, test_q, model_key="bert-base-mean")       # no retrieval training
display(lab.evaluate(C, TEST, {
    "BM25": bm25_test,
    "bert-base-uncased, mean pooling (no retrieval training)": bert_test,
    "bge-small-en-v1.5 (bi-encoder)": dense_test,
}))

,collection,split,system,queries,nDCG@10,MRR@10,R@10,R@100
0,scifact,test,BM25,300,0.6862,0.6506,0.8168,0.9209
1,scifact,test,"bert-base-uncased, mean pooling (no retrieval training)",300,0.1334,0.1149,0.2108,0.4818
2,scifact,test,bge-small-en-v1.5 (bi-encoder),300,0.7127,0.6822,0.8362,0.9417


**What is in the cache.** The next cell lists the embedding files for SciFact and prints the metadata stored next to the bge-small vectors.
`device` and `encode_s` record where and how fast the vectors were originally computed. In the cost table of Section 8, rows marked
`from_cache = True` show those original numbers.

**Code in the next cell.**

* `CFG.cache_dir / "embeddings"`: the folder of the second cache layer. Each `.json` file there has a `.npy` file of the same name next to it.
* `sci.fingerprint()` → the 16-character corpus hash that appears in the file names of this collection's embeddings.
* `lab.doc_embeddings(C, model_key=None)` → `(matrix, metadata)` for the default encoder. The matrix has one row per document, in the order of
  `sci.doc_ids`. The metadata is the content of the `.json` file plus `loaded_from_cache`.

In [8]:
# The embedding cache: one .npy (vectors) + one .json (metadata) per (model, revision, max length, corpus).
cache_files = sorted((CFG.cache_dir / "embeddings").glob("*.json"))
fingerprint = sci.fingerprint()
display(pd.DataFrame([{"file": f.stem, "for this corpus": fingerprint in f.stem,
                       "size MB": round(f.with_suffix(".npy").stat().st_size / 1e6, 1)} for f in cache_files]))
_, meta = lab.doc_embeddings(C)
print(json.dumps(meta, indent=1))

,file,for this corpus,size MB
0,bert-base-mean__86b5e093__len512__40bf9f3b7b849634,True,15.9
1,bert-base-mean__86b5e093__len512__a6bdd5328f55e4cf,False,11.2
2,bge-small__5c38ec7c__len512__40bf9f3b7b849634,True,8.0
3,bge-small__5c38ec7c__len512__a6bdd5328f55e4cf,False,5.6
4,distilbert-msmarco-v4__b2f66c95__len512__40bf9f3b7b849634,True,15.9
5,distilbert-msmarco-v4__b2f66c95__len512__a6bdd5328f55e4cf,False,11.2
6,distilbert-tas-b__b12d9352__len512__40bf9f3b7b849634,True,15.9
7,distilbert-tas-b__b12d9352__len512__a6bdd5328f55e4cf,False,11.2
8,e5-small__ffb93f3b__len512__40bf9f3b7b849634,True,8.0
9,e5-small__ffb93f3b__len512__a6bdd5328f55e4cf,False,5.6


{
 "model": "BAAI/bge-small-en-v1.5",
 "revision": "5c38ec7c405ec4b44b94cc5a9bb96e735b38267a",
 "max_seq_length": 512,
 "similarity": "cos",
 "n": 5183,
 "dim": 384,
 "device": "cuda",
 "model_load_s": 11.76,
 "encode_s": 8.05,
 "truncated_docs": 455,
 "loaded_from_cache": true
}


Each `.npy` file is a matrix with one row per document (5,183 × 384 for bge-small, 5,183 × 768 for bert-base), stored as 32-bit floats. The
corpus hash in the file name (`for this corpus = True`) shows which files belong to SciFact. Files for NFCorpus and for the other encoders of
Part B appear here once those sections have run. `truncated_docs` counts the abstracts longer than the 512-token limit.

Plain BERT reaches nDCG@10 ≈ 0.13, against 0.69 for BM25. Its R@100 ≈ 0.48 means that for about half of the claims the relevant abstract is
not even among the 100 most similar vectors. bge-small, built on the same kind of architecture, is above BM25 on every measure (by about 2.7
nDCG points; B8 shows that this gap is within sampling uncertainty for 300 claims). The architecture is not what makes a retriever. The
**training objective and data** are.

The next cell takes, *by rule*, the first test claim (by id) that bge-small answers at rank 1 while plain BERT misses it in its top 100. It
prints both top-3 lists and compares the cosine of the relevant abstract with the distribution of cosines between the claim and **all**
5,183 abstracts.

**Code in the next cell.**

* `analysis.first_ranks(runs, qrels)` → a table with one row per query and one column per run: the rank of the first relevant document, or
  `NaN` if none is in the run's top 100.
* The rule `(bge == 1) & bert is NaN`, then `.index[0]`, picks the first such claim by id.
* `lab.doc_embeddings(C, key)` → the cached document matrix of each encoder; `CFG.encoders[key]` → its `EncoderSpec`.
* `dense.encode_texts(spec, [text], "query", device, batch_size)` → the query vector; `dense.similarity_matrix(qv, emb, "cos", device)` → its
  cosine with all 5,183 documents. `sci.doc_index[doc_id]` gives the row of the relevant document in that vector.

In [9]:
# What does an untrained encoder consider "similar"? One claim picked by rule, plus cosine statistics.
from irlab import dense as dense_mod
fr_bert = analysis.first_ranks({"bert": bert_test, "bge": dense_test}, test_qrels)
example = fr_bert[(fr_bert["bge"] == 1) & fr_bert["bert"].isna()].index[0]
show.compare_systems(sci, {"bert-base-uncased (mean)": bert_test, "bge-small": dense_test},
                     example, test_q[example], test_qrels[example], k=3)
rel_doc = next(iter(test_qrels[example]))
for key in ("bert-base-mean", "bge-small"):
    spec = CFG.encoders[key]
    emb, _ = lab.doc_embeddings(C, key)
    qv = dense_mod.encode_texts(spec, [test_q[example]], "query", lab.device, 1)
    cos = dense_mod.similarity_matrix(qv, emb, "cos", lab.device)[0]
    print(f"{key:15s} cosine to relevant doc {cos[sci.doc_index[rel_doc]]:.3f} | median over all docs {np.median(cos):.3f} "
          f"| 99th percentile {np.percentile(cos, 99):.3f} | rank of relevant doc {int((cos > cos[sci.doc_index[rel_doc]]).sum()) + 1}")

QUERY 42: A high microerythrocyte count raises vulnerability to severe anemia in homozygous alpha (+)- thalassemia trait subjects.
judged relevant (1): 18174210 (label 1)
  [18174210] Increased Microerythrocyte Count in Homozygous α+-Thalassaemia Contributes to
        Protection against Severe Malarial Anaemia BACKGROUND The heritable haemoglobinopathy
        alpha(+)-thalassaemia is caused by the reduced synthesis of alpha-globin chains that form
        part of normal adult haemoglobin (Hb). Individuals homozygous for alpha(+)-thalassaemia have
        microcytosis and an increased erythrocyte ...
--- bert-base-uncased (mean): no relevant doc in its top 100
   1. [11457219] (unjudged)    0.870  Fumarase deficiency in dichorionic diamniotic twins. Fumarase deficiency is a rare autosomal recessive inborn error of metabolism of ...
   2. [18617259] (unjudged)    0.863  Research Letters We report a case of preclinical variant Creutzfeldt-Jakob disease (vCJD) in a patient who died from 

For claim 42, plain BERT gives the relevant abstract a cosine of 0.82, but the *median* cosine over all abstracts is already 0.76, and more than
200 abstracts score higher than the relevant one. Its top 3 (fumarase deficiency, Creutzfeldt-Jakob disease, neonatal anemia) share only
a vague medical register with the claim. bge-small spreads the scores: median 0.59, relevant abstract 0.84, at rank 1, followed by two
abstracts on thalassaemia and microcytosis. When unrelated pairs already score almost as high as the relevant one, ranking by cosine is
close to random. That is exactly what retrieval training fixes.

**Where BM25 and the dense model disagree.** Averages hide the fact that the two systems fail on *different* queries. The next cell computes,
for each test claim, the rank of the first relevant abstract under each system. It counts how often each one is better and prints the claims
with the largest gaps in both directions, chosen **by a fixed rule** (largest difference in reciprocal rank, ties by query id).

**Code in the next cell.**

* `analysis.first_ranks({"BM25": ..., "bge": ...}, test_qrels)` → first relevant rank per claim and system. A missing rank is replaced by
  101 (one past the depth) before taking reciprocals, so "not found" counts as worse than any rank.
* `analysis.pick_disagreements(fr, "BM25", "bge", n=3)` → `{"BM25 better": [...], "bge better": [...], "both miss top-10": [...]}`: the `n`
  claims with the largest difference in reciprocal first relevant rank in each direction, ties broken by query id, plus claims both
  systems miss in their top 10.
* `show.compare_systems(...)` as in Section 4, with `k=3`.

In [10]:
# Per-query disagreement between BM25 and the dense model, examples picked by rule.
fr = analysis.first_ranks({"BM25": bm25_test, "bge": dense_test}, test_qrels)
rr_bm, rr_de = 1 / fr["BM25"].fillna(101), 1 / fr["bge"].fillna(101)
print(f"first relevant ranked higher by BM25: {(rr_bm > rr_de).sum()}, by bge: {(rr_de > rr_bm).sum()}, "
      f"same rank: {(rr_bm == rr_de).sum()} (of {len(fr)} claims)")
picks = analysis.pick_disagreements(fr, "BM25", "bge", n=3)
print(json.dumps(picks))
for qid in picks["BM25 better"][:2] + picks["bge better"][:2]:
    show.compare_systems(sci, {"BM25": bm25_test, "bge-small": dense_test}, qid, test_q[qid], test_qrels[qid], k=3)

first relevant ranked higher by BM25: 58, by bge: 80, same rank: 162 (of 300 claims)
{"BM25 better": ["1363", "660", "674"], "bge better": ["452", "1088", "1368"], "both miss top-10": ["1", "1110", "1175"]}
QUERY 1363: Venules have a thinner or absent smooth layer compared to arterioles.
judged relevant (1): 8290953 (label 1)
  [8290953] Scaffold-based three-dimensional human fibroblast culture provides a structural matrix
        that supports angiogenesis in infarcted heart tissue. BACKGROUND We have developed
        techniques to implant angiogenic patches onto the epicardium over regions of infarcted
        cardiac tissue to stimulate revascularization of the damaged tissue. These experiments used
        a ...
--- BM25: first relevant at rank 1
 * 1. [8290953] (1)    7.435  Scaffold-based three-dimensional human fibroblast culture provides a structural matrix that supports angiogenesis in infarcted heart tissue. BACKGROUND We ...
   2. [1122279] (unjudged)    5.921  Endothelium-

BM25 ranks the first relevant abstract higher for 58 claims, bge-small for 80, and 162 claims are tied. The four examples show the main
mechanisms:

* **1363** (*Venules have a thinner or absent smooth layer compared to arterioles*). The relevant abstract's title is about fibroblast
  scaffolds for angiogenesis in infarcted heart tissue, and *venule*/*arteriole* occur only inside it. BM25 matches those rare words:
  rank 1. bge-small encodes the whole abstract as one 384-number vector dominated by its main topic, and the abstract is not in its top 100.
* **660** (*Ivermectin … onchocerciasis*). The single rare term `onchocerciasi` wins for BM25 (Section 4). bge-small prefers abstracts about
  drug therapy in general (rank 59).
* **452** (*Gene expression does not vary appreciably across genetically identical cells*). The relevant abstracts speak of *stochastic gene
  expression* and *cell-to-cell variation*: the same topic in different words. BM25 rank 64, bge-small rank 1.
* **1088** (*Silencing of Bcl2 …*). Tokenization: the claim's `bcl2` is a different index term from the abstract's *BCL-2* → `bcl` + `2`,
  so BM25 cannot match the gene name (rank 57). The dense model's WordPiece tokenizer splits both spellings into the same pieces (`bc`, `##l`, `2`),
so the vectors stay close (rank 1).

BM25 wins on rare exact strings, and the dense model wins on paraphrase and on surface variants of the same name. Neither dominates.

## 6 · Hybrid retrieval with reciprocal rank fusion

**Reciprocal rank fusion (RRF)** gives each document `Σ_systems 1 / (k + rank)`, summed over the systems that retrieved it in their top 100.
Only ranks are used, so BM25 scores (unbounded) never have to be compared with cosine similarities (in [−1, 1]). A document ranked high by
*either* system rises. `k = 60` (Cormack et al., 2009) keeps the difference between rank 1 and rank 5 small, so neither system dominates.
Section B6 varies `k` and the fusion depth, and compares RRF with score-based fusion.

**What the next cell does.** `lab.hybrid_run` asks the `lab` object for the BM25 run and the bge-small run of the same 300 claims. Both were
computed in Sections 4 and 5, so they come from memory (the third cache layer) and nothing is recomputed. It then adds the RRF contributions of
each document over the two lists and sorts the result. Fusion itself takes a few milliseconds.

The next cell fuses BM25 and bge-small and evaluates the result next to its two inputs.

**Code in the next cell.**

* `lab.hybrid_run(C, test_q, rrf_k=None, fusion_depth=None)` → a run. It takes `lab.bm25_run` and `lab.dense_run` of the same queries (both
  from memory here) and passes them to `fusion.rrf(runs, k, depth, out_depth)`: `k` is the RRF constant (default `CFG.rrf_k = 60`), `depth`
  how many documents of each input are fused (default 100) and `out_depth` how many fused documents are kept (100). Ties in the fused score
  are broken by document id.
* `analysis.first_ranks(test_runs, test_qrels)` → first relevant rank per claim for the three runs. `.fillna(101)` treats "not found" as rank
  101, and `best_input` is, per claim, the better of the BM25 and bge ranks.

In [11]:
# RRF fusion of BM25 and bge-small on the test claims.
rrf_test = lab.hybrid_run(C, test_q)
test_runs = {"BM25": bm25_test, "bge-small": dense_test, "RRF(BM25, bge-small)": rrf_test}
scifact_test_table = lab.evaluate(C, TEST, test_runs)
display(scifact_test_table)
fr3 = analysis.first_ranks(test_runs, test_qrels).fillna(101)
best_input = fr3[["BM25", "bge-small"]].min(axis=1)
print(f"RRF vs. the better of its two inputs (first relevant rank): better {int((fr3['RRF(BM25, bge-small)'] < best_input).sum())}, "
      f"same {int((fr3['RRF(BM25, bge-small)'] == best_input).sum())}, worse {int((fr3['RRF(BM25, bge-small)'] > best_input).sum())}")

,collection,split,system,queries,nDCG@10,MRR@10,R@10,R@100
0,scifact,test,BM25,300,0.6862,0.6506,0.8168,0.9209
1,scifact,test,bge-small,300,0.7127,0.6822,0.8362,0.9417
2,scifact,test,"RRF(BM25, bge-small)",300,0.7279,0.6952,0.8546,0.9683


RRF vs. the better of its two inputs (first relevant rank): better 21, same 208, worse 71


RRF is the best of the three runs on every measure. Its R@100 (0.968) is higher than either input's (0.921 and 0.942), because each system
contributes relevant abstracts that the other ranks below 100. The last line compares RRF, claim by claim, with whichever input was better
*for that claim*: RRF is better on 21 claims, equal on 208 and worse on 71. It can beat both inputs when the documents that push the relevant
abstract down are liked by only one of the systems. More often it lands between the two. Its average advantage comes from never needing to know
in advance which system will be right for a given claim, not from beating the better system on each claim.

**A worked example.** The next cell recomputes the fused score of the top documents of one claim by hand, from their ranks in the two input
lists, and checks the result against `lab.hybrid_run`. It makes the arithmetic visible: a document ranked by both systems collects two
contributions, a document ranked by one system only collects one.

**Code in the next cell.**

* `ranks[name][doc_id]` → the rank of a document in one input run, built from `run[qid]`, which is that query's list of `(doc id, score)`.
* `rrf_test[qid][:5]` → the fused top 5 with their RRF scores, as returned by `lab.hybrid_run`.
* `by_hand` = Σ 1/(`CFG.rrf_k` + rank) over the input runs that contain the document, written out in plain Python.

In [12]:
# Worked example: RRF scores of the fused top 5 for claim 1363, computed by hand from the two input ranks.
qid = "1363"
ranks = {name: {d: r for r, (d, _) in enumerate(run[qid], start=1)} for name, run in
         {"BM25": bm25_test, "bge-small": dense_test}.items()}
rows = []
for fused_rank, (d, fused_score) in enumerate(rrf_test[qid][:5], start=1):
    parts = {name: ranks[name].get(d) for name in ranks}
    by_hand = sum(1 / (CFG.rrf_k + r) for r in parts.values() if r is not None)
    rows.append({"RRF rank": fused_rank, "doc": d, "relevant": d in test_qrels[qid],
                 "BM25 rank": parts["BM25"], "bge rank": parts["bge-small"],
                 "1/(60+r_BM25) + 1/(60+r_bge)": round(by_hand, 5), "lab.hybrid_run score": round(fused_score, 5)})
rel = next(iter(test_qrels[qid]))
parts = {name: ranks[name].get(rel) for name in ranks}
print(f"claim {qid}: relevant abstract {rel} has BM25 rank {parts['BM25']}, bge rank {parts['bge-small']}, "
      f"RRF score {sum(1 / (CFG.rrf_k + r) for r in parts.values() if r is not None):.5f}")
display(pd.DataFrame(rows))

claim 1363: relevant abstract 8290953 has BM25 rank 1, bge rank None, RRF score 0.01639


,RRF rank,doc,relevant,BM25 rank,bge rank,1/(60+r_BM25) + 1/(60+r_bge),lab.hybrid_run score
0,1,23513718,False,4,9,0.03012,0.03012
1,2,25175997,False,15,1,0.02973,0.02973
2,3,79447,False,7,8,0.02963,0.02963
3,4,6417632,False,6,11,0.02924,0.02924
4,5,1122279,False,2,19,0.02879,0.02879


The two score columns agree, so the fused ranking is exactly the formula. Claim 1363 is the case from Section 5 where BM25 ranks the relevant
abstract first and bge-small does not retrieve it at all. Its fused score is therefore a single contribution, 1/(60 + 1) ≈ 0.0164. Every document in
the fused top 5 is retrieved by *both* systems. Even modest ranks in both lists, such as 2/(60 + 10) ≈ 0.0286, add up to more than one rank-1
contribution, so the relevant abstract is pushed below them.

The next cell shows, by rule, the ranks of the relevant abstract(s) before and after fusion: for the claim where RRF falls furthest behind its
better input, and for two claims where the inputs disagree most (from Section 5).

**Code in the next cell.**

* `show.rank_changes(runs, qid, rels)` → a table with one row per judged-relevant document of the query and one column per run: the rank of
  that document in the run, empty if it is not in the top 100.
* The first claim shown is the one where the RRF rank is furthest behind the better input (`gap` sorted, last entry). The other two are the
  first picks of `pick_disagreements` from Section 5.

In [13]:
# How fusion changes the rank of each relevant document: rule-picked claims.
gap = (fr3["RRF(BM25, bge-small)"] - best_input)
worst = gap.sort_values(kind="stable").index[-1]
for qid in [worst] + picks["BM25 better"][:1] + picks["bge better"][:1]:
    display(show.rank_changes(test_runs, qid, test_qrels[qid]).assign(claim=qid, text=test_q[qid][:70]))

,relevant doc,label,BM25,bge-small,"RRF(BM25, bge-small)",claim,text
0,3203590,1,None,56,99,913,PPAR-RXRs are inhibited by PPAR ligands.


,relevant doc,label,BM25,bge-small,"RRF(BM25, bge-small)",claim,text
0,8290953,1,1,None,16,1363,Venules have a thinner or absent smooth layer compared to arterioles.


,relevant doc,label,BM25,bge-small,"RRF(BM25, bge-small)",claim,text
0,12804937,1,NaN,1,16,452,Gene expression does not vary appreciably across genetically identical
1,464511,1,64.0,90,27,452,Gene expression does not vary appreciably across genetically identical


Each table lists the rank of the claim's relevant abstract(s) under BM25, bge-small and RRF (empty = not in the top 100). The three claims show
the cost of fusion:

* **1363**: BM25 ranks the relevant abstract 1st and bge-small does not retrieve it, so RRF ranks it **16th**. A document found by only one
  system receives a single contribution, 1/(60 + 1) ≈ 0.0164, while a document that *both* systems rank around 10th receives
  2/(60 + 10) ≈ 0.0286. Every document that both systems place in their top ~60 therefore outranks a one-system rank-1 hit.
* **452**: the same effect in the other direction (bge-small rank 1, BM25 rank 64 → RRF rank 16).
* **913**: the claim where RRF falls furthest behind its better input. Only bge-small retrieves the abstract, at rank 56, and RRF moves it
  to 99, just inside the list.

Fusion helps on average because it promotes documents that both systems agree on, and the same mechanism dilutes a strong signal that comes
from only one of them. RRF is used as the retrieval step for the LLM in Section 7.

## 7 · An LLM answers from retrieved passages

**Pipeline.** Question → RRF(BM25, bge-small) on SciFact → the top `n_sources = 3` abstracts (first 300 words of each, with their document ids)
→ a prompt that asks for an answer with an id in brackets after every sentence, and an exact refusal string if the passages are insufficient
→ `Qwen/Qwen2.5-1.5B-Instruct`, a 1.5B-parameter instruction-tuned decoder, with greedy decoding.

**Two different quality questions.**

* *Retrieval* is evaluated with qrels (Sections 4–6). Here we only print, for each retrieved passage, the qrels label of the claim the question
  was derived from and the original SciFact rationale label.
* *Answer quality* is whether each sentence is actually supported by the passage it cites. A program can check that cited ids were among the
  passages provided, and it can flag sentences without a citation. Whether a passage *supports* a sentence can only be judged by reading it.
  An instruction to cite or to abstain is a request to the model, not a guarantee.

**The four questions**, two that the collection can answer and two that it cannot. They were derived from SciFact test claims and written down,
with the expected faithful behaviour, before any of them was run:

| id | type | question |
|---|---|---|
| S1 | supported | Is ALDH1 expression associated with a poorer prognosis in breast cancer? |
| S2 | supported, evidence contradicts the premise | Are open-access articles cited less often than articles in traditional journals? |
| U1 | unsupported by the judged abstract (SciFact: no rationale) | Does cold exposure increase the recruitment of brown adipose tissue? |
| U2 | unsupported: outside the collection | In which year did the US FDA first approve a CRISPR-based therapy? |

**What `ask()` does, step by step.**

1. **Retrieve.** The question text is run through BM25, bge-small and RRF exactly like a claim (Sections 4–6), and the top `n_sources` documents
   are taken.
2. **Prepare passages.** Each document's title + abstract is cut to its first 300 words (most abstracts are shorter) and labelled with its document
   id, e.g. `[45638119] ALDH1 is a marker …`.
3. **Build the prompt.** A system message states the rules (use only the passages, cite an id after every sentence, or reply exactly
   `INSUFFICIENT EVIDENCE IN SOURCES`). The user message contains the passages and the question. The model's chat template turns both messages into
   the token sequence the model was fine-tuned on.
4. **Generate.** Greedy decoding picks the most probable next token at every step, up to 220 new tokens. There is no randomness, so the same input
   gives the same text on the same hardware and software.
5. **Check mechanically.** The ids in square brackets are extracted and compared with the ids that were provided. Sentences without a citation
   are listed, and the refusal string is detected.

For each source, `ask()` also prints the qrels label and the SciFact rationale label of the claim the question was derived from, so that the answer
can be compared with what the collection says.

**Caching.** The model weights (about 3 GB) are downloaded once into the Hugging Face cache (`~/.cache/huggingface/`) and loaded into memory once per
session. Answers are **not** cached: every call generates again.

The next cell loads the LLM and defines `ask()`. The 1.5B model needs about 3.5 GB of GPU memory in bfloat16, or about 7 GB of RAM in float32
on a CPU. On a slow CPU, `llm_id="Qwen/Qwen2.5-0.5B-Instruct"` (revision `7ae557604adf67be50417f59c2c2f167def9a775`) can be set in `CFG`, with
weaker instruction following.

**Code in the next cell.**

* `QUESTIONS`: `{id: {"id", "category", "source_claim", "question", "expected"}}`. `source_claim` is the SciFact test claim the question was
  derived from (`None` for U2), so its qrels and rationale labels can be shown next to the answer. `expected` is the faithful behaviour,
  written down before the run.
* `generation.load_llm(model_id, revision, device)` → `(tokenizer, model)`. On a GPU the weights are loaded in bfloat16, on a CPU in float32.
  It is memoised, so a second call returns the loaded model.
* `generation.SYSTEM_PROMPT`: the rules given to the model.
* `ask(item, n_sources=3, show_passages=True)` combines the steps listed above and returns a dict with the question, the source ids, the
  answer text and the mechanical checks. It uses:
  * `lab.hybrid_run(C, {key: question})` → an RRF run for this single question; its first `n_sources` documents are the sources;
  * `generation.make_sources(doc_ids, doc_text, max_words)` → a list of `Source(doc_id, text, truncated)`, the text cut to `max_words` words;
  * `generation.build_messages(question, sources)` → the chat messages `[{"role": "system", ...}, {"role": "user", ...}]`;
  * `generation.generate(tokenizer, model, messages, max_new_tokens, device)` → `{"text", "prompt_tokens", "new_tokens", "seconds"}`, greedy
    decoding;
  * `generation.citation_check(answer, sources)` → `{"abstained", "cited_ids", "cited_not_provided", "sentences_without_citation"}`.
  * `lab._time(...)` records the loading and generation times in the cost table of Section 8.

In [14]:
# Load the LLM and define the retrieval-augmented answering function.
QUESTIONS = {q["id"]: q for q in [
    {"id": "S1", "category": "supported", "source_claim": "53",
     "question": "Is ALDH1 expression associated with a poorer prognosis in breast cancer?",
     "expected": "yes, citing 45638119 (SciFact: SUPPORT for claim 53)"},
    {"id": "S2", "category": "supported (evidence contradicts the premise)", "source_claim": "129",
     "question": "Are articles published in open access format cited less often than articles in traditional journals?",
     "expected": "no, citing 27768226 (SciFact: CONTRADICT for claim 129)"},
    {"id": "U1", "category": "unsupported by the judged abstract (SciFact: no rationale)", "source_claim": "268",
     "question": "Does cold exposure increase the recruitment of brown adipose tissue?",
     "expected": "insufficient evidence, or only what the passages state; the judged abstract 970012 has no rationale"},
    {"id": "U2", "category": "unsupported: outside the collection", "source_claim": None,
     "question": "In which year did the US FDA first approve a CRISPR-based gene-editing therapy?",
     "expected": "abstain; no abstract covers this approval"},
]}
t0 = time.perf_counter()
tok, llm = generation.load_llm(CFG.llm_id, CFG.llm_revision, lab.device)
lab._time("setup", f"load LLM [{CFG.llm_id}]", C, time.perf_counter() - t0, None)
print(f"LLM ready on {lab.device} after {time.perf_counter() - t0:.1f} s")
print("SYSTEM PROMPT:", generation.SYSTEM_PROMPT)


def ask(item: dict, n_sources: int = CFG.llm_n_sources, show_passages: bool = True) -> dict:
    # Retrieve with RRF for the question text, show the passages, generate, and run the mechanical checks.
    if not 1 <= n_sources <= CFG.retrieval_depth:
        raise ValueError("n_sources must be between 1 and the retrieval depth")
    qkey = f"llm:{item['id']}"
    ranked = lab.hybrid_run(C, {qkey: item["question"]})[qkey][:n_sources]
    sources = generation.make_sources([d for d, _ in ranked], lambda d: sci.full_text(sci.doc_index[d]),
                                      CFG.llm_passage_words)
    claim = item.get("source_claim")
    rels = test_qrels.get(claim, {}) if claim else {}
    ev = evidence.get(claim, {}).get("evidence", {}) if claim else {}
    print("=" * 100)
    print(f"{item['id']} ({item['category']}): {item['question']}")
    if claim:
        print(f"derived from claim {claim}: {test_q[claim]!r}; judged relevant: {sorted(rels)}")
    for rank, s in enumerate(sources, start=1):
        print(f"  source {rank}: [{s.doc_id}] qrels of claim: {show.label_of(rels, s.doc_id)}; "
              f"SciFact rationale: {ev.get(s.doc_id, 'none') if claim else 'n/a'}; truncated: {s.truncated}")
        if show_passages:
            print("     ", " ".join(s.text.split()[:70]), "...")
    out = generation.generate(tok, llm, generation.build_messages(item["question"], sources),
                              CFG.llm_max_new_tokens, lab.device)
    lab._time("generate", f"LLM answer ({n_sources} sources)", C, out["seconds"], out["new_tokens"],
              note=f"{out['prompt_tokens']} prompt tokens; items = generated tokens")
    check = generation.citation_check(out["text"], sources)
    print(f"\nLLM ANSWER ({out['prompt_tokens']} prompt tokens, {out['new_tokens']} new tokens, {out['seconds']} s):")
    print(out["text"])
    print("\nmechanical checks:", json.dumps(check))
    return {"id": item["id"], "n_sources": n_sources, "question": item["question"],
            "sources": [s.doc_id for s in sources], "answer": out["text"], **check}

LLM ready on cuda after 1.0 s
SYSTEM PROMPT: You answer questions using ONLY the numbered source passages provided by the user. Every sentence of your answer must end with the id of the passage that supports it, in square brackets, for example [12345]. Do not use outside knowledge. If the passages do not contain the information needed, reply exactly: INSUFFICIENT EVIDENCE IN SOURCES.


Before generating, the next cell prints the **complete prompt** for S1 exactly as the model receives it, after the chat template has been applied.
The special markers (`<|im_start|>`, `<|im_end|>`) are how this model family separates the system, user and assistant turns.

**Code in the next cell.**

* The sources for S1 are built exactly as inside `ask()`.
* `tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)` → the prompt as one string, in this model's chat format.
  `add_generation_prompt=True` appends the marker that starts the assistant's turn, so the model continues with its answer.
* `tok(prompt_text)["input_ids"]` → the token ids; their number is the prompt length.

In [15]:
# The exact text the model receives for S1 (system rules + passages + question, after the chat template).
qkey = "llm:S1"
s1_ranked = lab.hybrid_run(C, {qkey: QUESTIONS["S1"]["question"]})[qkey][:CFG.llm_n_sources]
s1_sources = generation.make_sources([d for d, _ in s1_ranked], lambda d: sci.full_text(sci.doc_index[d]), CFG.llm_passage_words)
prompt_text = tok.apply_chat_template(generation.build_messages(QUESTIONS["S1"]["question"], s1_sources),
                                      tokenize=False, add_generation_prompt=True)
print(prompt_text[:1500], "\n[...]\n", prompt_text[-400:])
print(f"\nprompt length: {len(tok(prompt_text)['input_ids'])} tokens")

<|im_start|>system
You answer questions using ONLY the numbered source passages provided by the user. Every sentence of your answer must end with the id of the passage that supports it, in square brackets, for example [12345]. Do not use outside knowledge. If the passages do not contain the information needed, reply exactly: INSUFFICIENT EVIDENCE IN SOURCES.<|im_end|>
<|im_start|>user
Source passages:

[45638119] ALDH1 is a marker of normal and malignant human mammary stem cells and a predictor of poor clinical outcome. Application of stem cell biology to breast cancer research has been limited by the lack of simple methods for identification and isolation of normal and malignant stem cells. Utilizing in vitro and in vivo experimental systems, we show that normal and cancer human mammary epithelial cells with increased aldehyde dehydrogenase activity (ALDH) have stem/progenitor properties. These cells contain the subpopulation of normal breast epithelium with the broadest lineage diffe

**Code in the next cell.** `ask(QUESTIONS["S1"])` and `ask(QUESTIONS["S2"])` with the default `n_sources = 3`. Each call prints the sources with
their labels, the generated answer and the mechanical checks. The returned dicts are collected in `llm_log`.

In [16]:
# Two questions the collection can answer.
llm_log = [ask(QUESTIONS["S1"]), ask(QUESTIONS["S2"])]

S1 (supported): Is ALDH1 expression associated with a poorer prognosis in breast cancer?
derived from claim 53: 'ALDH1 expression is associated with poorer prognosis in breast cancer.'; judged relevant: ['45638119']
  source 1: [45638119] qrels of claim: 1; SciFact rationale: SUPPORT; truncated: False
      ALDH1 is a marker of normal and malignant human mammary stem cells and a predictor of poor clinical outcome. Application of stem cell biology to breast cancer research has been limited by the lack of simple methods for identification and isolation of normal and malignant stem cells. Utilizing in vitro and in vivo experimental systems, we show that normal and cancer human mammary epithelial cells with increased aldehyde dehydrogenase activity ...
  source 2: [24349992] qrels of claim: unjudged; SciFact rationale: none; truncated: True
      Oxidative stress in cancer associated fibroblasts drives tumor-stroma co-evolution: A new paradigm for understanding tumor metabolism, the field 


LLM ANSWER (1298 prompt tokens, 27 new tokens, 0.38 s):
Yes, ALDH1 expression is associated with a poorer prognosis in breast cancer. [45638119]

mechanical checks: {"abstained": false, "cited_ids": ["45638119"], "cited_not_provided": [], "sentences_without_citation": []}
S2 (supported (evidence contradicts the premise)): Are articles published in open access format cited less often than articles in traditional journals?
derived from claim 129: 'Articles published in open access format are less likely to be cited than traditional journals.'; judged relevant: ['27768226']
  source 1: [27768226] qrels of claim: 1; SciFact rationale: CONTRADICT; truncated: True
      Open Access Increases Citation Rate PLoS Biology publishes today a research article by Gunther Eysenbach that is not about biology. It is about citations. It provides robust evidence that open-access articles (OA articles) are more immediately recognized and cited than non-OA articles. As such, it adds objective support to t


LLM ANSWER (1148 prompt tokens, 28 new tokens, 0.25 s):
[27768226] Yes, open-access articles are more immediately recognized and cited than non-open-access articles.

mechanical checks: {"abstained": false, "cited_ids": ["27768226"], "cited_not_provided": [], "sentences_without_citation": []}


* **S1** answers *"Yes, ALDH1 expression is associated with a poorer prognosis in breast cancer. [45638119]"*. The cited abstract's title says
  ALDH1 is *"a predictor of poor clinical outcome"*, and SciFact labels it SUPPORT for the source claim. The sentence is **supported** by its
  citation.
* **S2** answers *"[27768226] Yes, open-access articles are more immediately recognized and cited than non-open-access articles."* The
  second half is a near-quote of the cited passage and is supported. But **"Yes" answers the question "are they cited *less*?" wrongly**:
  the passage says the opposite. The answer contradicts itself, carries the correct citation, and passes every mechanical check. Retrieval
  was perfect here (the relevant abstract is source 1), and the error is entirely in generation.

**Code in the next cell.** The same for U1 and U2, appended to `llm_log`.

In [17]:
# Two questions the collection cannot answer.
llm_log += [ask(QUESTIONS["U1"]), ask(QUESTIONS["U2"])]

U1 (unsupported by the judged abstract (SciFact: no rationale)): Does cold exposure increase the recruitment of brown adipose tissue?
derived from claim 268: 'Cold exposure increases BAT recruitment.'; judged relevant: ['970012']
  source 1: [29381091] qrels of claim: unjudged; SciFact rationale: none; truncated: False
      Brown and white adipose tissues: intrinsic differences in gene expression and response to cold exposure in mice Brown adipocytes dissipate energy, whereas white adipocytes are an energy storage site. We explored the plasticity of different white adipose tissue depots in acquiring a brown phenotype by cold exposure. By comparing cold-induced genes in white fat to those enriched in brown compared with white fat, at thermoneutrality we defined a "brite" transcription ...
  source 2: [14865329] qrels of claim: unjudged; SciFact rationale: none; truncated: False
      Brown Fat and Browning for the Treatment of Obesity and Related Metabolic Disorders Brown fat is a spec


LLM ANSWER (1057 prompt tokens, 10 new tokens, 0.11 s):
INSUFFICIENT EVIDENCE IN SOURCES

mechanical checks: {"abstained": true, "cited_ids": [], "cited_not_provided": [], "sentences_without_citation": []}


* **U1** answers *"Yes [29381091]."* The judged abstract of the source claim (970012) is **not** among the three passages (RRF ranks it 4th).
  The cited abstract, 29381091, is unjudged. It reports that cold exposure makes *white* fat depots acquire a brown ("brite") phenotype. That is
  related evidence, not the statement "cold increases recruitment of brown adipose tissue", so the answer is **partly supported at best**, and
  the one-word reply hides the difference. Two points follow: an unjudged passage can be useful, and a useful passage can still fail to
  justify the answer given.
* **U2** answers *"INSUFFICIENT EVIDENCE IN SOURCES"*. The passages are about postmarket drug safety and CRISPR experiments in rats, and none
  mentions an approval, so abstaining is correct. This is a single case. Section B7 shows the same model failing to abstain where it should,
  and abstaining where the evidence was present.

The next cell collects every answer sentence with its citations and the mechanical checks, and saves the answers to `results/`.

**Code in the next cell.**

* `generation.split_sentences(answer)` → the answer's sentences. A citation placed after the full stop (`"... cancer. [123]"`) is attached to
  the sentence before it.
* `generation.CITATION_RE.findall(sentence)` → the ids written in square brackets.
* `generation.ABSTAIN_MARKER` (`"INSUFFICIENT EVIDENCE"`) detects a refusal.
* `lab.save_table(df, filename)` writes the table to `results/`, and the full answers are written to `results/llm_answers.json`.

In [18]:
# Answer sentences, their citations, and the mechanical checks; saved for reproducibility.
rows = []
for r in llm_log:
    for s in generation.split_sentences(r["answer"]):
        rows.append({"question": r["id"], "sentence": s, "cited": generation.CITATION_RE.findall(s),
                     "cited id was provided": all(c not in r["cited_not_provided"] for c in generation.CITATION_RE.findall(s)),
                     "abstained": generation.ABSTAIN_MARKER in s.upper()})
answer_table = pd.DataFrame(rows)
display(answer_table)
lab.save_table(answer_table, "llm_answer_sentences.csv")
_ = (Path(CFG.results_dir) / "llm_answers.json").write_text(json.dumps(llm_log, indent=1))

,question,sentence,cited,cited id was provided,abstained
0,S1,"Yes, ALDH1 expression is associated with a poorer prognosis in breast cancer. [45638119]",[45638119],True,False
1,S2,"[27768226] Yes, open-access articles are more immediately recognized and cited than non-open-acc...",[27768226],True,False
2,U1,Yes [29381091].,[29381091],True,False
3,U2,INSUFFICIENT EVIDENCE IN SOURCES,[],True,True


saved results/llm_answer_sentences.csv


Every row passes the mechanical checks: all cited ids were among the passages. The problems in S2 (a wrong "Yes") and U1 (an over-general
claim) are invisible to these checks and are found only by reading the cited passages. Retrieval metrics and answer quality therefore have to
be assessed separately.

## 8 · Summary of results

The next cell prints the main SciFact test table and the **computational cost** of each step of this session, with setup, query and
generation separated, and saves both to `results/`. In the cost table, `setup` rows are paid once (loading models, indexing, encoding the
corpus). `from_cache = True` means the work was done earlier and loaded from disk, so the recorded seconds are those of the original
computation on the listed device. `query` rows cover all queries of a run (`ms/item` is the cost per query), and `generate` rows count
generated tokens.

**Code in the next cell.**

* `lab.timing_table()` → one row per timed step: `stage` (`setup`, `query` or `generate`), `component`, `collection`, `seconds`, `items`
  (documents, queries or generated tokens), `ms/item`, `device`, `from_cache` and `note`.
* `lab.save_table(...)` writes the tables to `results/`, and `results/environment.json` stores `CFG.summary()` with `lab.environment()`.

In [19]:
# Final tables, saved for reproducibility.
display(scifact_test_table)
lab.save_table(scifact_test_table, "scifact_test_main.csv")
timings = lab.timing_table()
with pd.option_context("display.max_rows", 200):
    display(timings[["stage", "component", "seconds", "items", "ms/item", "device", "from_cache", "note"]])
lab.save_table(timings, "timings_part_a.csv")
_ = (Path(CFG.results_dir) / "environment.json").write_text(json.dumps(
    {"config": CFG.summary(), "environment": lab.environment()}, indent=1, default=str))

,collection,split,system,queries,nDCG@10,MRR@10,R@10,R@100
0,scifact,test,BM25,300,0.6862,0.6506,0.8168,0.9209
1,scifact,test,bge-small,300,0.7127,0.6822,0.8362,0.9417
2,scifact,test,"RRF(BM25, bge-small)",300,0.7279,0.6952,0.8546,0.9683


saved results/scifact_test_main.csv


,stage,component,seconds,items,ms/item,device,from_cache,note
0,setup,load collection,0.032,NaN,NaN,cuda,False,
1,setup,"BM25 index (stop+stem, k1=1.2, b=0.75)",0.460,5183.0,0.09,cuda,False,
2,query,BM25 search,0.052,300.0,0.17,cuda,False,
3,setup,"encode corpus [bge-small, len 512]",8.050,5183.0,1.55,cuda,True,computed on cuda; model load 11.76 s
4,setup,load encoder [bge-small],0.805,NaN,NaN,cuda,False,
5,query,"dense encode+search [bge-small, cos]",0.237,300.0,0.79,cuda,False,
6,setup,"encode corpus [bert-base-mean, len 512]",21.120,5183.0,4.07,cuda,True,computed on cuda; model load 27.28 s
7,setup,load encoder [bert-base-mean],0.879,NaN,NaN,cuda,False,
8,query,"dense encode+search [bert-base-mean, cos]",0.191,300.0,0.64,cuda,False,
9,query,RRF fusion (k=60),0.008,300.0,0.03,cuda,False,


saved results/timings_part_a.csv


**What these numbers do and do not mean.**

* They are results on the **complete** BEIR SciFact test split (300 claims, all 5,183 documents), with settings fixed in advance or chosen on
  dev data. They are not results on the original SciFact *test* set, whose labels are not public.
* They are comparable *in kind* with BEIR tables but not identical in setup: tokenization, BM25 implementation, truncation and model versions
  differ.
* With 300 queries, differences of one or two nDCG points are within what a different sample of claims could produce (Section B8).
* Unjudged documents count as non-relevant, so a system that retrieves useful but unjudged abstracts is penalised.
* A score on SciFact answers "how well does this system find the cited abstract for a scientific claim?". A score on NFCorpus (B1) answers a
  different question. Raw numbers from different collections cannot be averaged or compared as a ranking of models.

---
# Part B · Further experiments

Each section needs only Sections 1–2 (setup and SciFact loading) and re-creates every run it uses through `lab`, which stores results by
their parameters. The sections can be run in any order.

**Protocol for every tuning experiment.** (1) Vary *one* setting while holding the others at the configuration of Section 1. (2) Choose
the value with the best **dev** nDCG@10. (3) Evaluate the a-priori value and the chosen value **once** on test. A test improvement found this
way is an honest estimate. A setting chosen by looking at test numbers would not be.

## B1 · A second collection: NFCorpus

**NFCorpus** (Boteva et al., ECIR 2016) links consumer-health pages of NutritionFacts.org to the PubMed articles they cite. BEIR's version has
3,633 abstracts, and its queries are page titles and topic tags such as *"Peoria"*, *"chickpeas"* or *"Do Fruit & Nut Bars Cause Weight Gain?"*.
Relevance is **graded**. In the NFCorpus documentation the highest level is a direct citation from the page, the middle level an indirect link
through another page, and the lowest level a shared topic tag. The BEIR qrels contain labels **2 and 1**. BEIR does not document exactly how the
three original levels were mapped onto them, so only the order 2 > 1 > unjudged is relied on.

**Why this second collection.** Compared with SciFact it changes the factors that drive retrieval behaviour while keeping the document type
(abstracts) comparable:

* **query style:** 1–3-word topics and catchy titles instead of precise 12-word claims;
* **relevance density:** about 38 judged documents per query, many judged only loosely (shared tag), instead of about 1;
* **graded labels**, so nDCG and MRR measure different things.

Licence: free for academic use according to the NFCorpus page; NutritionFacts.org content has its own terms of service. The dev split (324
queries) is used for choices and the test split (323) for reporting. The settings of Section 1 are used unchanged.

The next cell loads and describes NFCorpus, runs BM25, bge-small and RRF on the test queries, and computes the best achievable R@100: when a
query has more than 100 relevant documents, even a perfect system cannot reach R@100 = 1. The collection is downloaded into `data/beir/nfcorpus/`
the first time. NFCorpus has its own corpus hash, so its bge-small vectors are a separate file in `cache/embeddings/`. The first run encodes the
3,633 abstracts and later runs load them.

**Code in the next cell.** The same calls as in Part A, with `N = "nfcorpus"`: `lab.collection(N)`, `describe(nf)`, `lab.split_name(N, ...)`,
`lab.queries` / `lab.qrels`, then `lab.bm25_run`, `lab.dense_run` and `lab.hybrid_run` on the test queries, and `lab.evaluate`. The ceiling is
the mean over queries of min(1, 100 / number of relevant documents).

In [20]:
# B1: the same pipeline on NFCorpus (test split, settings unchanged from Section 1).
N = "nfcorpus"
nf = lab.collection(N)
display(describe(nf))
NF_DEV, NF_TEST = lab.split_name(N, "dev"), lab.split_name(N, "test")
nf_test_q, nf_test_qrels = lab.queries(N, NF_TEST), lab.qrels(N, NF_TEST)
nf_runs = {"BM25": lab.bm25_run(N, nf_test_q), "bge-small": lab.dense_run(N, nf_test_q)}
nf_runs["RRF(BM25, bge-small)"] = lab.hybrid_run(N, nf_test_q)
nf_test_table = lab.evaluate(N, NF_TEST, nf_runs)
display(nf_test_table)
lab.save_table(nf_test_table, "nfcorpus_test_main.csv")
ceiling = np.mean([min(1.0, 100 / sum(r >= 1 for r in v.values())) for v in nf_test_qrels.values()])
print(f"best achievable mean R@100 on NFCorpus test: {ceiling:.3f}")

,collection,split,role,queries,judgments,judged/query (mean),judged/query (median),label counts,query words (median),docs,doc words (median)
0,nfcorpus,dev,dev (tuning),324,11385,35.14,18.5,"{1: 10864, 2: 521}",2.0,3633,237.0
1,nfcorpus,test,test (final),323,12334,38.19,16.0,"{1: 11758, 2: 576}",2.0,3633,237.0
2,nfcorpus,train,unused,2590,110575,42.69,18.0,{1: 110575},2.0,3633,237.0


,collection,split,system,queries,nDCG@10,MRR@10,R@10,R@100
0,nfcorpus,test,BM25,323,0.3226,0.5289,0.1525,0.2464
1,nfcorpus,test,bge-small,323,0.3438,0.5302,0.1620,0.3110
2,nfcorpus,test,"RRF(BM25, bge-small)",323,0.3634,0.5684,0.1713,0.3120


saved results/nfcorpus_test_main.csv
best achievable mean R@100 on NFCorpus test: 0.965


The nDCG@10 values (about 0.32–0.36) are far lower than on SciFact. That reflects this *task and its qrels*, not weaker systems: many loosely
related abstracts are judged relevant, the short queries are under-specified, and R@100 is far below its ceiling of 0.965. MRR@10 (about
0.53–0.57) is much higher than nDCG@10, because finding *one* relevant document among 38 is easy, while ranking the top 10 like the ideal graded
ordering is hard. The *ordering* of systems is the same as on SciFact: BM25 < bge-small < RRF.

The next cell shows three queries chosen in advance for their form (a place name, a single food word, a pun title), followed by the largest
BM25-vs-dense disagreements picked by rule.

**Code in the next cell.**

* `qid_of` maps a query text back to its id, so queries can be selected by their text.
* `show.compare_systems(..., k=3, show_relevant_text=False)` prints the top 3 of each run without the text of the relevant documents (there
  can be dozens).
* `metrics.ndcg_at_k(ranked, rels, 10)` → nDCG@10 of one query, per run.
* `analysis.first_ranks` and `analysis.pick_disagreements(..., n=2)` as in Section 5.

In [21]:
# B1: NFCorpus examples: three queries chosen for their form, then rule-based disagreements.
qid_of = {text: q for q, text in nf_test_q.items()}
for text in ["Peoria", "chickpeas", "Unsafe at Any Feed"]:
    q = qid_of[text]
    show.compare_systems(nf, nf_runs, q, text, nf_test_qrels[q], k=3, show_relevant_text=False)
    print("per-query nDCG@10:",
          {k: round(metrics.ndcg_at_k([d for d, _ in r[q]], nf_test_qrels[q], 10), 3) for k, r in nf_runs.items()})
nf_fr = analysis.first_ranks({"BM25": nf_runs["BM25"], "bge": nf_runs["bge-small"]}, nf_test_qrels)
nf_picks = analysis.pick_disagreements(nf_fr, "BM25", "bge", n=2)
print(json.dumps(nf_picks))
for q in nf_picks["BM25 better"] + nf_picks["bge better"]:
    show.compare_systems(nf, {"BM25": nf_runs["BM25"], "bge-small": nf_runs["bge-small"]}, q, nf_test_q[q],
                         nf_test_qrels[q], k=3, show_relevant_text=False)

QUERY PLAIN-1827: Peoria
judged relevant (3): MED-4539 (label 1), MED-4540 (label 1), MED-4541 (label 1)
--- BM25: retrieved nothing (no query term occurs in the corpus)
--- bge-small: no relevant doc in its top 100
   1. [MED-1041] (unjudged)    0.557  Gastroenterology in ancient Egypt. Physicians in ancient Egypt devoted their care to disorders of individual organs. Notable among ...
   2. [MED-1003] (unjudged)    0.545  In Utero and Childhood Polybrominated Diphenyl Ether (PBDE) Exposures and Neurodevelopment in the CHAMACOS Study background: California children’s ...
   3. [MED-2404] (unjudged)    0.544  Increased Rate of Hospitalization for Diabetes and Residential Proximity of Hazardous Waste Sites Background Epidemiologic studies suggest that ...
--- RRF(BM25, bge-small): no relevant doc in its top 100
   1. [MED-1041] (unjudged)    0.016  Gastroenterology in ancient Egypt. Physicians in ancient Egypt devoted their care to disorders of individual organs. Notable among ...
   2. 

* **Peoria**: BM25 retrieves *nothing*, because the word occurs in no abstract. Its three judged documents (e.g. *Natural products as sources of
  new drugs*, *monacolin levels in red yeast rice*) are linked to the query only through NutritionFacts' page and tag structure. Every system
  scores 0. This is a property of how the qrels were built, not a model failure, and it holds for many 1-word tag queries.
* **chickpeas** (92 judged documents): all systems start with relevant abstracts, and RRF has the best nDCG@10 for this query.
* **Unsafe at Any Feed** (a pun on *Unsafe at Any Speed*): the label-2 document is *Public knowledge and beliefs about diarrheal disease*, and
  the label-1 document ranks the disease burden of food-borne pathogens. Not one content word is shared with the title. BM25 finds neither
  in its top 100, and bge-small finds one at rank 70.
* Rule-based picks: **rickets** and **ultra-processed foods** are BM25 wins on the exact term or phrase, while bge-small drifts to neighbouring
  topics (bone and diet, *processed meat*). **myelopathy** is the reverse: no abstract contains the word, so BM25 returns nothing, while
  bge-small ranks a case report of *subacute combined degeneration of the spinal cord* first. **prenatal vitamins**: BM25 is pulled to abstracts
  on *prenatal exposure* to pollutants (rank 49), and bge-small finds maternal B12 deficiency at rank 1.

Graded labels show up in the `(label)` column: a label-2 document at rank 1 counts twice as much in DCG as a label-1 document.

## B2 · SciFact claims grouped by measurable properties

Instead of hand-picking examples, three properties are computed for every test claim, the claims are grouped, and the mean nDCG@10 of each
system is reported per group. Examples from each group are then printed in an order fixed by rule.

* **overlap**: the share of the claim's index terms that occur in its relevant abstract(s). Low overlap is *vocabulary mismatch* (paraphrase,
  synonyms). This uses the qrels, so it is a **diagnostic**, not something a search engine could know.
* **min_df**: the document frequency of the rarest term. 0 means the term is unknown to the index (misspelling, unusual tokenization), and
  1–5 means a rare entity or piece of jargon.
* **words**: claim length.

Per-query nDCG@10 is coarse (with one relevant document it is 1, 0.63, 0.5, … or 0), so group means must be read with group sizes in mind.

**Code in the next cell.**

* `lab.per_query(C, TEST, run)` → one row per claim with nDCG@10, MRR@10, R@10 and R@100; `pq` holds one such table per system.
* `analysis.query_features(collection, queries, qrels, index)` → one row per claim: `words`, `query_terms` (number of distinct index terms),
  `overlap`, `min_df` and `oov_terms` (number of terms unknown to the index). It uses the BM25 index for tokenization and document frequencies.
* `analysis.bucket_table(per_query, feature, bins, labels, measure="nDCG@10")` → groups the claims by `feature` into the intervals given by
  `bins` (left-closed), named by `labels`, and reports the number of claims and each system's mean `measure` per group.

In [22]:
# B2: per-group effectiveness on the SciFact test claims.
b2_runs = {"BM25": lab.bm25_run(C, test_q), "bge-small": lab.dense_run(C, test_q)}
b2_runs["RRF"] = lab.hybrid_run(C, test_q)
pq = {k: lab.per_query(C, TEST, r) for k, r in b2_runs.items()}
feats = analysis.query_features(sci, test_q, test_qrels, lab.bm25_index(C))
for title, col, bins, labels in [
    ("lexical overlap with relevant abstract", "overlap", [0, 0.5, 0.8, 1.01], ["<0.5", "0.5–0.8", "≥0.8"]),
    ("rarest query term (document frequency)", "min_df", [0, 1, 6, 10**9], ["0 (unknown term)", "1–5 (rare)", "≥6"]),
    ("claim length in words", "words", [0, 9, 15, 1000], ["<9", "9–14", "≥15"]),
]:
    print(title)
    display(analysis.bucket_table(pq, feats[col], bins, labels))

lexical overlap with relevant abstract


,bucket,queries,BM25,bge-small,RRF
0,<0.5,98,0.339,0.454,0.445
1,0.5–0.8,125,0.808,0.789,0.818
2,≥0.8,77,0.931,0.918,0.941


rarest query term (document frequency)


,bucket,queries,BM25,bge-small,RRF
0,0 (unknown term),57,0.649,0.710,0.730
1,1–5 (rare),122,0.747,0.711,0.730
2,≥6,121,0.642,0.716,0.725


claim length in words


,bucket,queries,BM25,bge-small,RRF
0,<9,71,0.692,0.673,0.732
1,9–14,144,0.676,0.713,0.713
2,≥15,85,0.699,0.746,0.750


The groups show where each system's advantage comes from. In the low-overlap group (overlap < 0.5, 98 claims) bge-small is far ahead of BM25
(0.454 vs. 0.339): the dense model's gain is concentrated in vocabulary mismatch. In the rare-term group (df 1–5, 122 claims) BM25 is ahead
(0.747 vs. 0.711). RRF stays within 0.02 of the best system in every group. Claim length makes less difference than either property.

The next cell prints two claims from each group with the top 3 of BM25, bge-small and RRF. The rules are: the lowest overlap, rare terms
(smallest positive df), unknown terms, the longest and shortest claims, and claims for which **all** systems miss the top 10. Ties are
broken by query id.

**Code in the next cell.** `gallery` maps each group name to two claim ids, selected with pandas sorting on the features table (`qid` is the
tie-breaker). `analysis.first_ranks` finds the claims that every system misses in its top 10. `show.compare_systems(...)` prints each example.

In [23]:
# B2: two examples per group, chosen by fixed rules.
f = feats.assign(qid=feats.index)
gallery = {
    "lowest overlap (vocabulary mismatch)": f.sort_values(["overlap", "qid"]).index[:2],
    "rare term (1 <= df <= 5)": f[(f.min_df >= 1) & (f.min_df <= 5)].sort_values(["min_df", "qid"]).index[:2],
    "unknown term (df = 0)": f[f.oov_terms > 0].sort_values("qid").index[:2],
    "longest claims": f.sort_values(["words", "qid"], ascending=[False, True]).index[:2],
    "shortest claims": f.sort_values(["words", "qid"]).index[:2],
}
fr_all = analysis.first_ranks(b2_runs, test_qrels)
all_miss = fr_all[(fr_all.isna() | (fr_all > 10)).all(axis=1)].index
gallery["all systems miss the top 10"] = all_miss[:2]
for group, qids in gallery.items():
    print("\n" + "#" * 30, group, "#" * 30)
    for q in qids:
        print(f"features: {feats.loc[q].to_dict()}")
        show.compare_systems(sci, b2_runs, q, test_q[q], test_qrels[q], k=3, show_relevant_text=True)
print("\nnumber of test claims that every system misses in the top 10:", len(all_miss))


############################## lowest overlap (vocabulary mismatch) ##############################
features: {'words': 5.0, 'query_terms': 6.0, 'overlap': 0.0, 'min_df': 1.0, 'oov_terms': 0.0}
QUERY 1: 0-dimensional biomaterials show inductive properties.
judged relevant (1): 31715818 (label 1)
  [31715818] New opportunities: the use of nanotechnologies to manipulate and track stem cells.
        Nanotechnologies are emerging platforms that could be useful in measuring, understanding,
        and manipulating stem cells. Examples include magnetic nanoparticles and quantum dots for
        stem cell labeling and in vivo tracking; nanoparticles, carbon nanotubes, ...
--- BM25: no relevant doc in its top 100
   1. [18953920] (unjudged)    4.505  The Epithelial-Mesenchymal Transition Generates Cells with Properties of Stem Cells The epithelial-mesenchymal transition (EMT) is a key developmental ...
   2. [10608397] (unjudged)    4.475  High-performance neuroprosthetic control by an indivi

The gallery makes the group statistics concrete. In the lowest-overlap claims, the relevant abstract uses different vocabulary from the claim
(for claim 1, *0-dimensional biomaterials* against an abstract on nanotechnologies for stem cells), and BM25 cannot reach it. Rare-term claims
are typically decided by a single gene, drug or disease name. Among the shortest claims, 1099 (*Statins decrease blood cholesterol*) and 1100
(*Statins increase blood cholesterol*) share their relevant abstract and are ranked almost identically. The claims that every system misses are
mostly ones whose cited abstract addresses the claim only indirectly, such as a global estimate of childhood underweight cited for a claim about
perinatal mortality.

**Negation and contradiction pairs.** SciFact contains pairs of claims that differ by a negation or an opposite outcome and share the same
cited abstract. A retrieval system *should* return the same abstract for both, because it addresses both. Deciding which claim it supports is a
separate step. The next cell shows the rank of the shared abstract for three such pairs.

**Code in the next cell.** For each claim of a pair: its relevant document (`next(iter(test_qrels[q]))`, the single judged document), its
SciFact label from `evidence`, and its rank in each run (`ranked.index(d) + 1`, empty if not retrieved).

In [24]:
# B2: claim pairs that contradict each other but share their relevant abstract.
pairs = [("51", "53"), ("56", "57"), ("1278", "1279")]
rows = []
for a, b in pairs:
    for q in (a, b):
        d = next(iter(test_qrels[q]))
        row = {"claim": q, "text": test_q[q], "relevant doc": d,
               "SciFact label": evidence[q]["evidence"].get(d, "none (NEI)")}
        for k, r in b2_runs.items():
            ranked = [x for x, _ in r[q]]
            row[k] = ranked.index(d) + 1 if d in ranked else None
        rows.append(row)
display(pd.DataFrame(rows))

,claim,text,relevant doc,SciFact label,BM25,bge-small,RRF
0,51,ALDH1 expression is associated with better breast cancer outcomes.,45638119,CONTRADICT,1,1,1
1,53,ALDH1 expression is associated with poorer prognosis in breast cancer.,45638119,SUPPORT,1,1,1
2,56,APOE4 expression in iPSC-derived neurons increases AlphaBeta production and tau phosphorylation ...,4709641,SUPPORT,1,1,1
3,57,"APOE4 expression in iPSC-derived neurons increases AlphaBeta production and tau phosphorylation,...",4709641,CONTRADICT,1,1,1
4,1278,The treatment of cancer patients with co-IR blockade does not cause any adverse autoimmune events.,11335781,none (NEI),10,1,2
5,1279,The treatment of cancer patients with co-IR blockade precipitates adverse autoimmune events.,11335781,none (NEI),8,1,2


Opposite claims get the shared abstract at the same or nearly the same rank from every system. Claims 51 and 53 (*better* vs. *poorer* breast
cancer outcomes with ALDH1) both retrieve abstract 45638119 at rank 1, although SciFact labels it CONTRADICT for one and SUPPORT for the other.
Retrieval finds the *topic*, not the *truth*.

## B3 · Controlled query reformulations

To isolate the effect of *wording*, the information need is kept fixed and only the formulation changes. The next cell defines 23
reformulations of 8 test claims: **paraphrases**, **synonym/abbreviation substitutions** (e.g. *CX3CR1* → *fractalkine receptor*,
*chenodeoxycholic acid* → *CDCA*), **keyword** versions, **long natural-language questions**, a **spelling correction** and a **broader entity**.
Each variant **inherits the qrels of its source claim**. That assumption holds for paraphrases but is weaker for short keyword queries, where
other, unjudged abstracts may also be relevant. The variants were written before any of them was run, and all are reported.

The next cell runs every variant and reports, for each one, the rank of the first relevant abstract under each system (empty = not in the top
100), next to the original claim. The variants are new queries, so their BM25 scores and query embeddings are computed fresh. The document
vectors come from the embedding cache, so no abstract is re-encoded.

**Code in the next cell.**

* `VAR`: a list of `{"source": claim id, "kind": type of reformulation, "text": the new query}`.
* `var_q` / `var_qrels`: the variants as queries and qrels, with ids `var<i>:<source>`; each variant gets the qrels of its source claim.
  `both_q` / `both_qrels` add the original claims, so originals and variants are retrieved and evaluated in the same runs.
* `lab.bm25_run`, `lab.dense_run`, `lab.hybrid_run` on `both_q`. This set of query texts is new, so it gets new runs. The document side
  (BM25 index, cached embeddings) is reused.
* `analysis.first_ranks(b3_runs, both_qrels)` → first relevant rank per query and system, rearranged into one row per variant.

In [25]:
# B3: run every reformulation; rank of the inherited relevant abstract per system.
VAR = [
    {"source": '42', "kind": 'paraphrase',
     "text": 'Having many small red blood cells makes people with two copies of the alpha-plus thalassaemia variant more prone to serious anaemia.'},
    {"source": '42', "kind": 'keywords',
     "text": 'microerythrocyte thalassemia anemia'},
    {"source": '42', "kind": 'long question',
     "text": 'I read that in people who carry two copies of the alpha-plus thalassemia trait, having a high number of very small red blood cells might make severe anemia more likely; is there research on whether that is true?'},
    {"source": '118', "kind": 'paraphrase',
     "text": 'Changes to intestinal bacteria caused by antimicrobial drugs make the host more susceptible to C. diff infection.'},
    {"source": '118', "kind": 'keywords',
     "text": 'antibiotics microbiome C. difficile'},
    {"source": '118', "kind": 'long question',
     "text": 'When a patient takes antibiotics and the bacteria living in their intestines change as a result, does that make it easier for Clostridium difficile to cause an infection?'},
    {"source": '129', "kind": 'paraphrase',
     "text": 'Freely available papers receive fewer citations than subscription-only publications.'},
    {"source": '129', "kind": 'keywords',
     "text": 'open access citations'},
    {"source": '129', "kind": 'long question',
     "text": 'If a researcher publishes a paper so that anyone can read it for free, will it end up being cited less often than a paper in a traditional paywalled journal?'},
    {"source": '163', "kind": 'paraphrase',
     "text": 'Weight-loss operations improve psychological well-being.'},
    {"source": '163', "kind": 'keywords',
     "text": 'bariatric surgery mental health'},
    {"source": '163', "kind": 'long question',
     "text": 'For patients with severe obesity who undergo weight loss surgery, do conditions such as depression or anxiety tend to get better afterwards?'},
    {"source": '219', "kind": 'synonym (entity)',
     "text": 'The fractalkine receptor expressed by T helper 2 lymphocytes dampens inflammation of the airways.'},
    {"source": '219', "kind": 'keywords',
     "text": 'CX3CR1 Th2 airway inflammation'},
    {"source": '219', "kind": 'long question',
     "text": 'Does the chemokine receptor CX3CR1, when it is present on Th2 helper T cells, reduce inflammation in the airways of the lung, for example in asthma?'},
    {"source": '248', "kind": 'spelling corrected',
     "text": 'Chenodeoxycholic acid treatment increases whole-body energy expenditure.'},
    {"source": '248', "kind": 'synonym (abbreviation)',
     "text": "Taking the bile acid CDCA raises the body's total energy use."},
    {"source": '248', "kind": 'keywords',
     "text": 'chenodeoxycholic acid energy expenditure'},
    {"source": '274', "kind": 'paraphrase',
     "text": 'Combining smoking-cessation drugs with varenicline gives better one-year quit rates than varenicline by itself.'},
    {"source": '274', "kind": 'keywords',
     "text": 'varenicline bupropion abstinence'},
    {"source": '236', "kind": 'paraphrase',
     "text": 'In songbirds, body cells determine their own sex independently of hormones.'},
    {"source": '236', "kind": 'keywords',
     "text": 'cell-autonomous sex somatic cells'},
    {"source": '236', "kind": 'broader entity',
     "text": 'Cell autonomous sex determination in somatic cells occurs in birds.'},
]
var_q = {f"var{i}:{v['source']}": v["text"] for i, v in enumerate(VAR)}
var_qrels = {f"var{i}:{v['source']}": test_qrels[v["source"]] for i, v in enumerate(VAR)}
orig_q = {v["source"]: test_q[v["source"]] for v in VAR}
both_q, both_qrels = {**orig_q, **var_q}, {**{q: test_qrels[q] for q in orig_q}, **var_qrels}
b3_runs = {"BM25": lab.bm25_run(C, both_q), "bge-small": lab.dense_run(C, both_q)}
b3_runs["RRF"] = lab.hybrid_run(C, both_q)
fr_var = analysis.first_ranks(b3_runs, both_qrels)
rows = []
for src in dict.fromkeys(v["source"] for v in VAR):
    rows.append({"source": src, "kind": "ORIGINAL", "text": test_q[src], **fr_var.loc[src].to_dict()})
    for i, v in enumerate(VAR):
        if v["source"] == src:
            rows.append({"source": src, "kind": v["kind"], "text": v["text"], **fr_var.loc[f"var{i}:{src}"].to_dict()})
variants_table = pd.DataFrame(rows)
with pd.option_context("display.max_colwidth", 70):
    display(variants_table)
lab.save_table(variants_table, "query_variants_first_relevant_rank.csv")
summary = []
for kind in ["ORIGINAL"] + sorted({v["kind"] for v in VAR}):
    sub = variants_table[variants_table.kind == kind]
    summary.append({"kind": kind, "n": len(sub), **{k: f"{(sub[k] <= 10).sum()}/{len(sub)}" for k in b3_runs}})
print("relevant abstract in the top 10, by reformulation type:")
display(pd.DataFrame(summary))

,source,kind,text,BM25,bge-small,RRF
0,42,ORIGINAL,A high microerythrocyte count raises vulnerability to severe anemi...,2.0,1.0,1.0
1,42,paraphrase,Having many small red blood cells makes people with two copies of ...,1.0,1.0,1.0
2,42,keywords,microerythrocyte thalassemia anemia,18.0,2.0,5.0
3,42,long question,I read that in people who carry two copies of the alpha-plus thala...,NaN,1.0,15.0
4,118,ORIGINAL,Antibiotic induced alterations in the gut microbiome reduce resist...,1.0,1.0,1.0
5,118,paraphrase,Changes to intestinal bacteria caused by antimicrobial drugs make ...,15.0,1.0,4.0
6,118,keywords,antibiotics microbiome C. difficile,1.0,1.0,1.0
7,118,long question,When a patient takes antibiotics and the bacteria living in their ...,2.0,1.0,2.0
8,129,ORIGINAL,Articles published in open access format are less likely to be cit...,1.0,1.0,1.0
9,129,paraphrase,Freely available papers receive fewer citations than subscription-...,1.0,1.0,1.0


saved results/query_variants_first_relevant_rank.csv
relevant abstract in the top 10, by reformulation type:


,kind,n,BM25,bge-small,RRF
0,ORIGINAL,8,8/8,8/8,8/8
1,broader entity,1,1/1,1/1,1/1
2,keywords,8,7/8,8/8,8/8
3,long question,5,4/5,5/5,4/5
4,paraphrase,6,4/6,6/6,6/6
5,spelling corrected,1,1/1,1/1,1/1
6,synonym (abbreviation),1,1/1,1/1,1/1
7,synonym (entity),1,1/1,1/1,1/1


The original claims are all found at rank 1–2 by every system, so these are "easy" needs, and the variants test robustness.

* bge-small keeps the relevant abstract in its top 3 for **all 23** variants.
* BM25 loses it from the top 10 in 4 cases: two paraphrases (118 → rank 15, 163 → rank 24), the keyword version of 42 (rank 18) and the long
  question for 42, where it is not in the top 100. That question drops the rare word *microerythrocyte* and spells *thalassemia* the American way,
  while the abstract's title says *Thalassaemia*. The paraphrase of 42 happens to use *thalassaemia*, and BM25 ranks it first.
* Variants expected to hurt BM25 did **not**: the spelling correction, *CDCA* and *fractalkine receptor* all leave BM25 at rank 1, because the
  remaining terms (*bile acid … energy*, *airway inflammation*, *T helper*) still identify the abstract.
* RRF stays in the top 10 everywhere except the long question for 42 (rank 15).

With 1 to 8 variants per type, these are cases that illustrate mechanisms, not general conclusions.

## B4 · Encoder choice and similarity function

**Encoders compared** (all BERT-family, all used through Sentence Transformers; revisions pinned in `irlab/config.py`):

| key | model | params | trained for retrieval on | similarity | prompt |
|---|---|---|---|---|---|
| bert-base-mean | google-bert/bert-base-uncased | 110M | – (MLM + NSP only) | cosine | – |
| distilbert-msmarco-v4 | sentence-transformers/msmarco-distilbert-base-v4 | 66M | MS MARCO | cosine | – |
| distilbert-tas-b | sentence-transformers/msmarco-distilbert-base-tas-b | 66M | MS MARCO (TAS-Balanced distillation) | dot product | – |
| bge-small | BAAI/bge-small-en-v1.5 | 33M | large mixed text-pair collections | cosine | query instruction |
| e5-small | intfloat/e5-small-v2 | 33M | CCPairs (incl. S2ORC title–abstract pairs), then MS MARCO, NQ, NLI | cosine | `query: ` / `passage: ` |

The two MS MARCO models were trained only on web-search data, so on SciFact/NFCorpus they are clean zero-shot transfers. E5's pre-training data
includes S2ORC title–abstract pairs, and SciFact's abstracts come from S2ORC, so E5 may have *seen these documents* (not the claims or qrels).
BGE's training data is not documented at the document level. **Selection rule:** highest mean dev nDCG@10 over the two collections.

The next cell encodes both corpora with every model, evaluates on **dev** (for selection) and on **test** (reported for completeness and *not*
used for the choice), and records the encoding time per document. Each model × corpus pair is one embedding file (10 files). On the first run this
is the slowest cell of the notebook, because every model must encode every abstract. Afterwards all vectors are loaded from `cache/embeddings/`,
and the cost table shows the original encoding times with `from_cache = True`. The dev and test queries of a collection share the same document
vectors, since only the queries differ.

**Code in the next cell.**

* `CFG.encoders`: the five `EncoderSpec`s of the table above. The loop calls `lab.dense_run(collection, queries, model_key=key)` for each one
  on the dev and the test queries of both collections.
* `lab.evaluate(...).assign(role=role)` adds a column saying whether the row is dev or test, and `pd.concat` stacks the tables.
* `pivot_table(index="system", columns=[...], values="nDCG@10")` rearranges nDCG@10 into one row per system.
* The selection takes the mean of the two dev columns per encoder and sorts it.
* The last table shows the `encode corpus` rows of `lab.timing_table()`: seconds and ms per document for every model and corpus.

In [26]:
# B4: five encoders on dev (selection) and test (report), both collections.
rows = []
for coll_name in ["scifact", "nfcorpus"]:
    for role in ("dev", "test"):
        split = lab.split_name(coll_name, role)
        qs = lab.queries(coll_name, split)
        runs = {"BM25 (reference)": lab.bm25_run(coll_name, qs)}
        for key in CFG.encoders:
            runs[key] = lab.dense_run(coll_name, qs, model_key=key)
        rows.append(lab.evaluate(coll_name, split, runs).assign(role=role))
enc_table = pd.concat(rows, ignore_index=True)
display(enc_table.pivot_table(index="system", columns=["collection", "role"], values="nDCG@10").round(3))
dev_mean = (enc_table[enc_table.role == "dev"].pivot_table(index="system", columns="collection", values="nDCG@10")
            .drop(index="BM25 (reference)").mean(axis=1).sort_values(ascending=False))
print("mean dev nDCG@10 over the two collections:\n", dev_mean.round(4).to_string())
print("selected:", dev_mean.index[0])
lab.save_table(enc_table, "encoders_dev_test.csv")
t = lab.timing_table()
display(t[t.component.str.startswith("encode corpus")][["component", "collection", "seconds", "items", "ms/item", "note"]])

collection            nfcorpus        scifact       
role                       dev   test     dev   test
system                                              
BM25 (reference)         0.290  0.323   0.692  0.686
bert-base-mean           0.043  0.043   0.124  0.133
bge-small                0.328  0.344   0.747  0.713
distilbert-msmarco-v4    0.260  0.268   0.556  0.529
distilbert-tas-b         0.293  0.320   0.646  0.643
e5-small                 0.322  0.326   0.708  0.688

mean dev nDCG@10 over the two collections:
 system
bge-small                0.5374
e5-small                 0.5150
distilbert-tas-b         0.4696
distilbert-msmarco-v4    0.4080
bert-base-mean           0.0836
selected: bge-small
saved results/encoders_dev_test.csv


,component,collection,seconds,items,ms/item,note
3,"encode corpus [bge-small, len 512]",scifact,8.05,5183.0,1.55,computed on cuda; model load 11.76 s
6,"encode corpus [bert-base-mean, len 512]",scifact,21.12,5183.0,4.07,computed on cuda; model load 27.28 s
31,"encode corpus [bge-small, len 512]",nfcorpus,6.12,3633.0,1.68,computed on cuda; model load 0.75 s
40,"encode corpus [distilbert-msmarco-v4, len 512]",scifact,10.66,5183.0,2.06,computed on cuda; model load 21.5 s
43,"encode corpus [distilbert-tas-b, len 512]",scifact,10.66,5183.0,2.06,computed on cuda; model load 20.18 s
47,"encode corpus [e5-small, len 512]",scifact,8.13,5183.0,1.57,computed on cuda; model load 15.4 s
54,"encode corpus [bert-base-mean, len 512]",nfcorpus,16.07,3633.0,4.42,computed on cuda; model load 0.78 s
57,"encode corpus [distilbert-msmarco-v4, len 512]",nfcorpus,8.17,3633.0,2.25,computed on cuda; model load 0.47 s
59,"encode corpus [distilbert-tas-b, len 512]",nfcorpus,8.18,3633.0,2.25,computed on cuda; model load 0.46 s
63,"encode corpus [e5-small, len 512]",nfcorpus,6.25,3633.0,1.72,computed on cuda; model load 1.54 s


bge-small has the highest mean dev nDCG@10 (0.537), followed by e5-small (0.515), TAS-B (0.470), msmarco-distilbert-v4 (0.408) and plain BERT
(0.084). The order is the same on dev and on test in both collections, so the dev set is a reliable guide here. The two MS MARCO-only models fall
below BM25 on SciFact, and TAS-B roughly ties with it on NFCorpus. This reproduces the BEIR paper's finding that dense retrievers trained on one
domain often lose to BM25 on another. Our TAS-B test scores (SciFact 0.643, NFCorpus 0.320) match BEIR Table 2 (0.643, 0.319), a useful check
that the pipeline is set up correctly. Encoding cost per document grows with model size and is paid once per corpus.

**Similarity function.** A model should be used with the similarity it was trained for. TAS-B was trained with the **dot product**, so its vector
norms carry information. bge-small outputs normalised vectors, so for it cosine and dot product give the same ranking (a sanity check). The next
cell switches only the similarity function.

**Code in the next cell.** `lab.dense_run(..., model_key=..., similarity="dot" | "cos")`. The `similarity` parameter overrides the model's
own function. `"cos"` divides the query and document vectors by their length before the dot product, while `"dot"` uses them as they are. The
document matrix is the same in both cases.

In [27]:
# B4: similarity function; only the scoring function changes, the vectors are the same.
sim_rows = []
for coll_name in ["scifact", "nfcorpus"]:
    split = lab.split_name(coll_name, "dev")
    qs = lab.queries(coll_name, split)
    runs = {"tas-b, dot (as trained)": lab.dense_run(coll_name, qs, model_key="distilbert-tas-b", similarity="dot"),
            "tas-b, cosine": lab.dense_run(coll_name, qs, model_key="distilbert-tas-b", similarity="cos"),
            "bge-small, cosine (as trained)": lab.dense_run(coll_name, qs, model_key="bge-small", similarity="cos"),
            "bge-small, dot": lab.dense_run(coll_name, qs, model_key="bge-small", similarity="dot")}
    sim_rows.append(lab.evaluate(coll_name, split, runs))
display(pd.concat(sim_rows, ignore_index=True))

,collection,split,system,queries,nDCG@10,MRR@10,R@10,R@100
0,scifact,train,"tas-b, dot (as trained)",809,0.6459,0.6182,0.7616,0.9046
1,scifact,train,"tas-b, cosine",809,0.5363,0.5051,0.6701,0.8694
2,scifact,train,"bge-small, cosine (as trained)",809,0.7469,0.7136,0.8694,0.9652
3,scifact,train,"bge-small, dot",809,0.7469,0.7136,0.8694,0.9652
4,nfcorpus,dev,"tas-b, dot (as trained)",324,0.2932,0.5070,0.1388,0.2747
5,nfcorpus,dev,"tas-b, cosine",324,0.2548,0.4648,0.1095,0.2468
6,nfcorpus,dev,"bge-small, cosine (as trained)",324,0.3278,0.5172,0.1498,0.3151
7,nfcorpus,dev,"bge-small, dot",324,0.3278,0.5172,0.1498,0.3151


No new encoding happens here: both similarity functions are applied to the same cached vectors, and only the final scoring step differs.
For bge-small the two rows are identical, as normalised vectors imply. For TAS-B, switching to cosine discards the vector length the model was
trained to use, and costs 11 nDCG points on SciFact dev (0.646 → 0.536) and 4 on NFCorpus dev (0.293 → 0.255). An embedding model must be used
with the similarity function it was trained with.

## B5 · BM25 preprocessing and parameters

**Step 1, preprocessing**, at the default k1 = 1.2, b = 0.75: stopword removal on/off × Snowball stemming on/off.
**Step 2, k1 and b**, with the dev-best preprocessing, on a small grid: k1 ∈ {0.5, 0.9, 1.2, 1.5, 2.0} (from almost binary term presence to
strong term-frequency influence) × b ∈ {0.3, 0.5, 0.75, 0.9} (from weak to strong length normalisation). Twenty points are enough to see the
shape of the surface without encouraging overfitting to dev. Both steps run on each collection's dev split, and the a-priori and chosen
settings are then evaluated once on test, with a paired bootstrap interval for the difference.

**Code in the next cell.**

* `Preprocessing(stopwords=..., stem=...)`: the four combinations. `.label` gives names such as `stop+stem` or `nostop+nostem`. Each
  combination builds its own `BM25Index`.
* `lab.bm25_run(collection, queries, prep=p, k1=..., b=...)`: one run per setting. Each new (preprocessing, k1, b) combination builds a new
  index, which takes under a second.
* `metrics.evaluate(run, qrels, ("nDCG@10",))` computes only nDCG@10, enough for the grid.
* The best grid point is chosen by dev nDCG@10. Ties are broken by the smaller k1, then the smaller b.
* `metrics.paired_bootstrap(a, b, n, seed)` takes two per-query score series (same queries) and returns the mean of `b − a`, its 95%
  percentile interval over `n` resamples of the queries, and the number of wins, losses and ties.

In [28]:
# B5: BM25 preprocessing and k1/b, chosen on dev, reported on test.
b6_final = []
for coll_name in ["scifact", "nfcorpus"]:
    dev_s, test_s = lab.split_name(coll_name, "dev"), lab.split_name(coll_name, "test")
    qd, qt = lab.queries(coll_name, dev_s), lab.queries(coll_name, test_s)
    preps = [Preprocessing(stopwords=s, stem=m) for s in (True, False) for m in (True, False)]
    prep_table = lab.evaluate(coll_name, dev_s, {p.label: lab.bm25_run(coll_name, qd, prep=p) for p in preps})
    print(f"{coll_name} dev: preprocessing (k1=1.2, b=0.75)"); display(prep_table)
    best_prep = preps[int(prep_table["nDCG@10"].idxmax())]
    grid = pd.DataFrame([{"k1": k1, "b": b, "nDCG@10": metrics.evaluate(
        lab.bm25_run(coll_name, qd, prep=best_prep, k1=k1, b=b), lab.qrels(coll_name, dev_s), ("nDCG@10",))["nDCG@10"]}
        for k1 in (0.5, 0.9, 1.2, 1.5, 2.0) for b in (0.3, 0.5, 0.75, 0.9)])
    print(f"{coll_name} dev: nDCG@10 over k1 (rows) × b (columns), preprocessing {best_prep.label}")
    display(grid.pivot(index="k1", columns="b", values="nDCG@10").round(4))
    bk = grid.sort_values(["nDCG@10", "k1", "b"], ascending=[False, True, True]).iloc[0]
    tuned = lab.bm25_run(coll_name, qt, prep=best_prep, k1=float(bk.k1), b=float(bk.b))
    default = lab.bm25_run(coll_name, qt)
    res = lab.evaluate(coll_name, test_s, {"a priori: stop+stem, k1=1.2, b=0.75": default,
                                           f"dev-chosen: {best_prep.label}, k1={bk.k1}, b={bk.b}": tuned})
    ci = metrics.paired_bootstrap(lab.per_query(coll_name, test_s, default)["nDCG@10"],
                                  lab.per_query(coll_name, test_s, tuned)["nDCG@10"], CFG.bootstrap_samples, CFG.seed)
    print(f"{coll_name} test: tuned minus default nDCG@10:", {k: round(v, 4) for k, v in ci.items()})
    b6_final.append(res)
b6_table = pd.concat(b6_final, ignore_index=True)
display(b6_table)
lab.save_table(b6_table, "bm25_tuning_test.csv")

scifact dev: preprocessing (k1=1.2, b=0.75)


,collection,split,system,queries,nDCG@10,MRR@10,R@10,R@100
0,scifact,train,stop+stem,809,0.6925,0.6615,0.8110,0.9314
1,scifact,train,stop+nostem,809,0.6664,0.6329,0.7908,0.9075
2,scifact,train,nostop+stem,809,0.6951,0.6666,0.8073,0.9314
3,scifact,train,nostop+nostem,809,0.6670,0.6347,0.7884,0.9075


scifact dev: nDCG@10 over k1 (rows) × b (columns), preprocessing nostop+stem


b,0.30,0.50,0.75,0.90
k1,,,,
0.5,0.6827,0.6873,0.6874,0.6870
0.9,0.6915,0.6935,0.6926,0.6931
1.2,0.6913,0.6934,0.6951,0.6931
1.5,0.6903,0.6931,0.6950,0.6927
2.0,0.6886,0.6903,0.6920,0.6915


scifact test: tuned minus default nDCG@10: {'mean diff': -0.0015, '95% CI low': -0.009, '95% CI high': 0.0059, 'wins': 14, 'losses': 13, 'ties': 273}


nfcorpus dev: preprocessing (k1=1.2, b=0.75)


... (2 outputs left out)

b,0.30,0.50,0.75,0.90
k1,,,,
0.5,0.2828,0.2847,0.2848,0.2839
0.9,0.2865,0.2876,0.2886,0.2877
1.2,0.2901,0.2914,0.2907,0.2888
1.5,0.2906,0.2918,0.2916,0.2883
2.0,0.2902,0.2912,0.2900,0.2890


nfcorpus test: tuned minus default nDCG@10: {'mean diff': 0.0042, '95% CI low': 0.0005, '95% CI high': 0.0081, 'wins': 56, 'losses': 52, 'ties': 215}


,collection,split,system,queries,nDCG@10,MRR@10,R@10,R@100
0,scifact,test,"a priori: stop+stem, k1=1.2, b=0.75",300,0.6862,0.6506,0.8168,0.9209
1,scifact,test,"dev-chosen: nostop+stem, k1=1.2, b=0.75",300,0.6847,0.6515,0.8082,0.9213
2,nfcorpus,test,"a priori: stop+stem, k1=1.2, b=0.75",323,0.3226,0.5289,0.1525,0.2464
3,nfcorpus,test,"dev-chosen: nostop+stem, k1=1.5, b=0.5",323,0.3268,0.5340,0.1574,0.2460


saved results/bm25_tuning_test.csv


Stemming matters: it adds 2.6 nDCG points on SciFact dev and 2.3 on NFCorpus dev, because it merges word forms such as *cite/cited/citation*.
Stopword removal does not help, and keeping the stopwords is marginally better on both collections. The k1 × b surface is flat (SciFact dev
0.683–0.695, NFCorpus dev 0.283–0.292), so BM25 is insensitive to these parameters on abstracts of fairly uniform length. On test, the
dev-chosen settings change nDCG@10 by −0.0015 [−0.009, +0.006] on SciFact and +0.004 [+0.0005, +0.008] on NFCorpus: tuning buys a fraction of a
point at most, and the a-priori defaults were already reasonable.

## B6 · Fusion settings

Three one-at-a-time experiments on dev, each against RRF(k = 60, depth 100):

1. **RRF k** ∈ {1, 10, 30, 60, 100, 300}. Small k gives large weight to the very top ranks, and large k gives a flatter contribution.
2. **Fusion depth**: how many documents of each input list are fused (10, 100, 1000). With depth 10, a document only enters if one system puts it
   in its top 10.
3. **Score-based fusion** instead of rank fusion: α·minmax(BM25) + (1−α)·minmax(cosine), α ∈ {0.1, 0.3, 0.5, 0.7, 0.9}.

The dev-best of each family is then evaluated once on test next to the a-priori RRF.

**Code in the next cell.**

* `lab.hybrid_run(collection, queries, rrf_k=k)` and `lab.hybrid_run(..., fusion_depth=d)`: RRF with one parameter changed. `fusion_depth`
  sets how many documents each input contributes. For depth 1000 the input runs are recomputed with depth 1000, and the fused list is
  still cut to 100.
* `minmax_fusion(a, b, alpha)` (`irlab/fusion.py`): per query, each run's scores are rescaled to [0, 1] (minimum → 0, maximum → 1) and combined as
  `alpha · BM25 + (1 − alpha) · dense`. A document missing from one list gets 0 for that list.
* The dev-best `k` and `alpha` are read back from the `system` labels of the dev table, then evaluated on test.

In [29]:
# B6: fusion parameters, one at a time, on dev; selected settings once on test.
from irlab.fusion import minmax_fusion
b7_final = []
for coll_name in ["scifact", "nfcorpus"]:
    dev_s, test_s = lab.split_name(coll_name, "dev"), lab.split_name(coll_name, "test")
    qd, qt = lab.queries(coll_name, dev_s), lab.queries(coll_name, test_s)
    exp = {f"RRF k={k}": lab.hybrid_run(coll_name, qd, rrf_k=k) for k in (1, 10, 30, 60, 100, 300)}
    exp.update({f"RRF k=60, fusion depth {d}": lab.hybrid_run(coll_name, qd, fusion_depth=d) for d in (10, 1000)})
    b_dev, d_dev = lab.bm25_run(coll_name, qd), lab.dense_run(coll_name, qd)
    exp.update({f"min-max α={a}": minmax_fusion(b_dev, d_dev, a) for a in (0.1, 0.3, 0.5, 0.7, 0.9)})
    t = lab.evaluate(coll_name, dev_s, exp)
    print(f"{coll_name} dev"); display(t)
    best_k = int(t[t.system.str.match(r"RRF k=\d+$")].sort_values("nDCG@10", ascending=False).iloc[0].system.split("=")[1])
    best_a = float(t[t.system.str.startswith("min-max")].sort_values("nDCG@10", ascending=False).iloc[0].system.split("=")[1])
    test_runs_b7 = {"RRF k=60 (a priori)": lab.hybrid_run(coll_name, qt),
                    f"RRF k={best_k} (dev-chosen)": lab.hybrid_run(coll_name, qt, rrf_k=best_k),
                    f"min-max α={best_a} (dev-chosen)": minmax_fusion(lab.bm25_run(coll_name, qt), lab.dense_run(coll_name, qt), best_a)}
    b7_final.append(lab.evaluate(coll_name, test_s, test_runs_b7))
b7_table = pd.concat(b7_final, ignore_index=True)
display(b7_table)
lab.save_table(b7_table, "fusion_test.csv")

scifact dev


,collection,split,system,queries,nDCG@10,MRR@10,R@10,R@100
0,scifact,train,RRF k=1,809,0.7581,0.7249,0.8787,0.9701
1,scifact,train,RRF k=10,809,0.7559,0.7210,0.8797,0.9701
2,scifact,train,RRF k=30,809,0.7508,0.7192,0.8658,0.9701
3,scifact,train,RRF k=60,809,0.7496,0.7188,0.8633,0.9701
4,scifact,train,RRF k=100,809,0.7487,0.7183,0.8609,0.9701
5,scifact,train,RRF k=300,809,0.7478,0.7180,0.8590,0.9701
6,scifact,train,"RRF k=60, fusion depth 10",809,0.7524,0.7174,0.8756,0.9024
7,scifact,train,"RRF k=60, fusion depth 1000",809,0.7497,0.7186,0.8633,0.9726
8,scifact,train,min-max α=0.1,809,0.7557,0.7224,0.8758,0.9726
9,scifact,train,min-max α=0.3,809,0.7665,0.7340,0.8813,0.9726


nfcorpus dev


,collection,split,system,queries,nDCG@10,MRR@10,R@10,R@100
0,nfcorpus,dev,RRF k=1,324,0.3326,0.5409,0.1488,0.3196
1,nfcorpus,dev,RRF k=10,324,0.3390,0.5394,0.1527,0.3196
2,nfcorpus,dev,RRF k=30,324,0.3403,0.5379,0.1534,0.3196
3,nfcorpus,dev,RRF k=60,324,0.3393,0.5380,0.1526,0.3196
4,nfcorpus,dev,RRF k=100,324,0.3383,0.5367,0.1523,0.3196
5,nfcorpus,dev,RRF k=300,324,0.3379,0.5363,0.1521,0.3196
6,nfcorpus,dev,"RRF k=60, fusion depth 10",324,0.3305,0.5370,0.1462,0.1775
7,nfcorpus,dev,"RRF k=60, fusion depth 1000",324,0.3385,0.5386,0.1517,0.3206
8,nfcorpus,dev,min-max α=0.1,324,0.3330,0.5262,0.1512,0.3237
9,nfcorpus,dev,min-max α=0.3,324,0.3414,0.5433,0.1557,0.3219


,collection,split,system,queries,nDCG@10,MRR@10,R@10,R@100
0,scifact,test,RRF k=60 (a priori),300,0.7279,0.6952,0.8546,0.9683
1,scifact,test,RRF k=1 (dev-chosen),300,0.7284,0.6951,0.8496,0.9683
2,scifact,test,min-max α=0.3 (dev-chosen),300,0.7289,0.6993,0.8429,0.9667
3,nfcorpus,test,RRF k=60 (a priori),323,0.3634,0.5684,0.1713,0.3120
4,nfcorpus,test,RRF k=30 (dev-chosen),323,0.3638,0.5685,0.1714,0.3120
5,nfcorpus,test,min-max α=0.3 (dev-chosen),323,0.3605,0.5597,0.1686,0.3170


saved results/fusion_test.csv


* **RRF k.** Dev prefers a small k on SciFact (k = 1: 0.758 vs. 0.750 at k = 60) and k = 30 on NFCorpus, but on test the chosen k changes
  nDCG@10 by less than 0.001. RRF is robust to k, which is the usual argument for leaving it at 60.
* **Fusion depth.** Fusing only the top 10 of each list cuts R@100 sharply (SciFact dev 0.902 vs. 0.970, NFCorpus dev 0.178 vs. 0.320),
  because documents that only one system finds deeper in its list are lost. Depth 1000 adds almost nothing over 100.
* **Score-based fusion.** α = 0.3 (more weight on the dense scores) is best on dev in both collections. On test it ties RRF on SciFact
  (0.729 vs. 0.728) and is slightly worse on NFCorpus (0.361 vs. 0.363). It needs a tuned weight to match RRF, so RRF's merit is robustness
  without tuning rather than a higher score.

## B7 · How many sources should the LLM see?

The four questions of Section 7 are asked with 1, 3 and 5 retrieved passages. *Only* `n_sources` changes: same retrieval, same prompt, same greedy
decoding. Fewer passages mean that a relevant one may be missing. More passages mean longer prompts and more chances for the model to cite
something off-topic. The table records, for every run, whether a judged-relevant document of the source claim was among the passages (a
*retrieval* fact) and what the model did (an *answer* fact).

**Code in the next cell.** `ask(item, n_sources=n, show_passages=False)` from Section 7, for n = 1, 3, 5. Nothing else changes. The table
combines the returned fields (`sources`, `abstained`, `cited_ids`, `cited_not_provided`, `answer`) with a retrieval fact: whether a judged-relevant
document of the source claim was among the sources.

In [30]:
# B7: number of sources given to the LLM (1, 3, 5); retrieval facts and answer facts side by side.
b8 = []
for qid_ in ("S1", "S2", "U1", "U2"):
    item = QUESTIONS[qid_]
    rels = test_qrels.get(item["source_claim"], {}) if item["source_claim"] else {}
    for n in (1, 3, 5):
        r = ask(item, n_sources=n, show_passages=False)
        b8.append({"question": qid_, "n_sources": n,
                   "judged-relevant doc among sources": any(d in rels for d in r["sources"]) if rels else "n/a",
                   "abstained": r["abstained"], "cited": r["cited_ids"], "cited not provided": r["cited_not_provided"],
                   "answer": r["answer"][:160]})
b8_table = pd.DataFrame(b8)
with pd.option_context("display.max_colwidth", 160):
    display(b8_table)
lab.save_table(b8_table, "llm_n_sources.csv")

S1 (supported): Is ALDH1 expression associated with a poorer prognosis in breast cancer?
derived from claim 53: 'ALDH1 expression is associated with poorer prognosis in breast cancer.'; judged relevant: ['45638119']
  source 1: [45638119] qrels of claim: 1; SciFact rationale: SUPPORT; truncated: False

LLM ANSWER (333 prompt tokens, 13 new tokens, 0.11 s):
Yes, [45638119].

mechanical checks: {"abstained": false, "cited_ids": ["45638119"], "cited_not_provided": [], "sentences_without_citation": []}
S1 (supported): Is ALDH1 expression associated with a poorer prognosis in breast cancer?
derived from claim 53: 'ALDH1 expression is associated with poorer prognosis in breast cancer.'; judged relevant: ['45638119']
  source 1: [45638119] qrels of claim: 1; SciFact rationale: SUPPORT; truncated: False
  source 2: [24349992] qrels of claim: unjudged; SciFact rationale: none; truncated: True
  source 3: [20886584] qrels of claim: unjudged; SciFact rationale: none; truncated: False



LLM ANSWER (1298 prompt tokens, 27 new tokens, 0.25 s):
Yes, ALDH1 expression is associated with a poorer prognosis in breast cancer. [45638119]

mechanical checks: {"abstained": false, "cited_ids": ["45638119"], "cited_not_provided": [], "sentences_without_citation": []}
S1 (supported): Is ALDH1 expression associated with a poorer prognosis in breast cancer?
derived from claim 53: 'ALDH1 expression is associated with poorer prognosis in breast cancer.'; judged relevant: ['45638119']
  source 1: [45638119] qrels of claim: 1; SciFact rationale: SUPPORT; truncated: False
  source 2: [24349992] qrels of claim: unjudged; SciFact rationale: none; truncated: True
  source 3: [20886584] qrels of claim: unjudged; SciFact rationale: none; truncated: False
  source 4: [16734530] qrels of claim: unjudged; SciFact rationale: none; truncated: False
  source 5: [15113221] qrels of claim: unjudged; SciFact rationale: none; truncated: False



LLM ANSWER (1958 prompt tokens, 27 new tokens, 0.28 s):
Yes, ALDH1 expression is associated with a poorer prognosis in breast cancer. [45638119]

mechanical checks: {"abstained": false, "cited_ids": ["45638119"], "cited_not_provided": [], "sentences_without_citation": []}
S2 (supported (evidence contradicts the premise)): Are articles published in open access format cited less often than articles in traditional journals?
derived from claim 129: 'Articles published in open access format are less likely to be cited than traditional journals.'; judged relevant: ['27768226']
  source 1: [27768226] qrels of claim: 1; SciFact rationale: CONTRADICT; truncated: True

LLM ANSWER (506 prompt tokens, 10 new tokens, 0.09 s):
INSUFFICIENT EVIDENCE IN SOURCES

mechanical checks: {"abstained": true, "cited_ids": [], "cited_not_provided": [], "sentences_without_citation": []}
S2 (supported (evidence contradicts the premise)): Are articles published in open access format cited less often than articles


LLM ANSWER (1148 prompt tokens, 28 new tokens, 0.25 s):
[27768226] Yes, open-access articles are more immediately recognized and cited than non-open-access articles.

mechanical checks: {"abstained": false, "cited_ids": ["27768226"], "cited_not_provided": [], "sentences_without_citation": []}
S2 (supported (evidence contradicts the premise)): Are articles published in open access format cited less often than articles in traditional journals?
derived from claim 129: 'Articles published in open access format are less likely to be cited than traditional journals.'; judged relevant: ['27768226']
  source 1: [27768226] qrels of claim: 1; SciFact rationale: CONTRADICT; truncated: True
  source 2: [4688277] qrels of claim: unjudged; SciFact rationale: none; truncated: False
  source 3: [10577574] qrels of claim: unjudged; SciFact rationale: none; truncated: False
  source 4: [16979690] qrels of claim: unjudged; SciFact rationale: none; truncated: True
  source 5: [28017005] qrels of claim: u


LLM ANSWER (1898 prompt tokens, 67 new tokens, 0.58 s):
[27768226] Yes, open-access articles gain a significant citation advantage over non-open-access articles within the same time frame. Specifically, they are twice as likely to be cited 4 to 10 months after publication and almost three times as likely between 10 and 16 months.

mechanical checks: {"abstained": false, "cited_ids": ["27768226"], "cited_not_provided": [], "sentences_without_citation": ["Specifically, they are twice as likely to be cited 4 to 10 months after publication and almost three times as likely between 10 and 16 months."]}
U1 (unsupported by the judged abstract (SciFact: no rationale)): Does cold exposure increase the recruitment of brown adipose tissue?
derived from claim 268: 'Cold exposure increases BAT recruitment.'; judged relevant: ['970012']
  source 1: [29381091] qrels of claim: unjudged; SciFact rationale: none; truncated: False

LLM ANSWER (367 prompt tokens, 12 new tokens, 0.1 s):
Yes [29381091].

me


LLM ANSWER (976 prompt tokens, 12 new tokens, 0.12 s):
Yes [29381091].

mechanical checks: {"abstained": false, "cited_ids": ["29381091"], "cited_not_provided": [], "sentences_without_citation": []}
U1 (unsupported by the judged abstract (SciFact: no rationale)): Does cold exposure increase the recruitment of brown adipose tissue?
derived from claim 268: 'Cold exposure increases BAT recruitment.'; judged relevant: ['970012']
  source 1: [29381091] qrels of claim: unjudged; SciFact rationale: none; truncated: False
  source 2: [14865329] qrels of claim: unjudged; SciFact rationale: none; truncated: False
  source 3: [4319174] qrels of claim: unjudged; SciFact rationale: none; truncated: False
  source 4: [970012] qrels of claim: 1; SciFact rationale: none; truncated: False
  source 5: [20532591] qrels of claim: unjudged; SciFact rationale: none; truncated: False



LLM ANSWER (1451 prompt tokens, 24 new tokens, 0.24 s):
Yes, cold exposure increases the recruitment of brown adipose tissue. [29381091]

mechanical checks: {"abstained": false, "cited_ids": ["29381091"], "cited_not_provided": [], "sentences_without_citation": []}
U2 (unsupported: outside the collection): In which year did the US FDA first approve a CRISPR-based gene-editing therapy?
  source 1: [3578380] qrels of claim: unjudged; SciFact rationale: n/a; truncated: True

LLM ANSWER (624 prompt tokens, 10 new tokens, 0.1 s):
INSUFFICIENT EVIDENCE IN SOURCES

mechanical checks: {"abstained": true, "cited_ids": [], "cited_not_provided": [], "sentences_without_citation": []}
U2 (unsupported: outside the collection): In which year did the US FDA first approve a CRISPR-based gene-editing therapy?
  source 1: [3578380] qrels of claim: unjudged; SciFact rationale: n/a; truncated: True
  source 2: [432261] qrels of claim: unjudged; SciFact rationale: n/a; truncated: False
  source 3: [13296399


LLM ANSWER (1057 prompt tokens, 10 new tokens, 0.11 s):
INSUFFICIENT EVIDENCE IN SOURCES

mechanical checks: {"abstained": true, "cited_ids": [], "cited_not_provided": [], "sentences_without_citation": []}
U2 (unsupported: outside the collection): In which year did the US FDA first approve a CRISPR-based gene-editing therapy?
  source 1: [3578380] qrels of claim: unjudged; SciFact rationale: n/a; truncated: True
  source 2: [432261] qrels of claim: unjudged; SciFact rationale: n/a; truncated: False
  source 3: [13296399] qrels of claim: unjudged; SciFact rationale: n/a; truncated: False
  source 4: [14174055] qrels of claim: unjudged; SciFact rationale: n/a; truncated: False
  source 5: [47018050] qrels of claim: unjudged; SciFact rationale: n/a; truncated: False

LLM ANSWER (1399 prompt tokens, 10 new tokens, 0.13 s):
INSUFFICIENT EVIDENCE IN SOURCES

mechanical checks: {"abstained": true, "cited_ids": [], "cited_not_provided": [], "sentences_without_citation": []}


,question,n_sources,judged-relevant doc among sources,abstained,cited,cited not provided,answer
0,S1,1,True,False,[45638119],[],"Yes, [45638119]."
1,S1,3,True,False,[45638119],[],"Yes, ALDH1 expression is associated with a poorer prognosis in breast cancer. [45638119]"
2,S1,5,True,False,[45638119],[],"Yes, ALDH1 expression is associated with a poorer prognosis in breast cancer. [45638119]"
3,S2,1,True,True,[],[],INSUFFICIENT EVIDENCE IN SOURCES
4,S2,3,True,False,[27768226],[],"[27768226] Yes, open-access articles are more immediately recognized and cited than non-open-access articles."
5,S2,5,True,False,[27768226],[],"[27768226] Yes, open-access articles gain a significant citation advantage over non-open-access articles within the same time frame. Specifically, they are ..."
6,U1,1,False,False,[29381091],[],Yes [29381091].
7,U1,3,False,False,[29381091],[],Yes [29381091].
8,U1,5,True,False,[29381091],[],"Yes, cold exposure increases the recruitment of brown adipose tissue. [29381091]"
9,U2,1,n/a,True,[],[],INSUFFICIENT EVIDENCE IN SOURCES


saved results/llm_n_sources.csv


* **S1** is answered correctly with 1, 3 and 5 passages (with one passage, only "Yes, [45638119].").
* **S2** **abstains with one passage**, although that passage directly contradicts the question's premise. With 3 and 5 passages it gives the
  self-contradictory "Yes, open-access articles are … cited more". With 5 it adds correct figures from the passage in a second sentence that has
  no citation, which the mechanical check flags.
* **U1** answers "Yes" in all three settings and cites the unjudged white-fat abstract 29381091, even with 5 passages that include the judged
  abstract 970012.
* **U2** abstains in all three settings.

More context did not make the answers more faithful, and the refusal instruction was ignored once (U1) and applied wrongly once (S2 with one
passage). With four questions these are case studies, not an evaluation of the model. A proper answer-quality evaluation needs many questions,
a fixed rubric and more than one annotator.

## B8 · Paired bootstrap intervals

For two systems run on the same queries, the queries are resampled with replacement 10,000 times (seed fixed), and the 2.5% and 97.5%
percentiles of the mean nDCG@10 difference are reported. The interval describes the uncertainty due to *which queries happened to be sampled*.
It does not cover other sources of variation (different collections, judging bias), and the comparisons are not corrected for multiple
testing. `wins/losses/ties` count queries.

**Code in the next cell.** `lab.per_query(...)["nDCG@10"]` gives one score per test query and system. `metrics.paired_bootstrap(a, b,
CFG.bootstrap_samples, CFG.seed)` resamples the query ids 10,000 times with a NumPy generator seeded with 13. It returns `mean diff` (b − a),
`95% CI low`, `95% CI high`, `wins`, `losses` and `ties`.

In [31]:
# B8: paired bootstrap of the main comparisons on the test splits of both collections.
rows = []
for coll_name in ["scifact", "nfcorpus"]:
    test_s = lab.split_name(coll_name, "test")
    qt = lab.queries(coll_name, test_s)
    r = {"BM25": lab.bm25_run(coll_name, qt), "bge-small": lab.dense_run(coll_name, qt)}
    r["RRF"] = lab.hybrid_run(coll_name, qt)
    pqs = {k: lab.per_query(coll_name, test_s, v)["nDCG@10"] for k, v in r.items()}
    for a, b in [("BM25", "bge-small"), ("bge-small", "RRF"), ("BM25", "RRF")]:
        ci = metrics.paired_bootstrap(pqs[a], pqs[b], CFG.bootstrap_samples, CFG.seed)
        rows.append({"collection": coll_name, "comparison": f"{b} minus {a}", **{k: round(v, 4) for k, v in ci.items()}})
boot = pd.DataFrame(rows)
display(boot)
lab.save_table(boot, "bootstrap_test.csv")

,collection,comparison,mean diff,95% CI low,95% CI high,wins,losses,ties
0,scifact,bge-small minus BM25,0.0265,-0.0046,0.0575,71,50,179
1,scifact,RRF minus bge-small,0.0152,-0.0066,0.0360,57,38,205
2,scifact,RRF minus BM25,0.0417,0.0205,0.0625,74,23,203
3,nfcorpus,bge-small minus BM25,0.0212,0.0010,0.0415,121,90,112
4,nfcorpus,RRF minus bge-small,0.0196,0.0064,0.0332,115,81,127
5,nfcorpus,RRF minus BM25,0.0408,0.0266,0.0556,122,59,142


saved results/bootstrap_test.csv


On SciFact (300 claims), neither bge-small vs. BM25 (+0.027) nor RRF vs. bge-small (+0.015) has an interval that excludes 0: with this many
queries the two differences cannot be told apart from sampling noise. On NFCorpus (323 queries), both bge-small vs. BM25 (+0.021) and RRF vs.
bge-small (+0.020) have intervals that exclude 0. RRF vs. BM25 excludes 0 on both collections (SciFact +0.042 [+0.021, +0.063], NFCorpus
+0.041 [+0.027, +0.056]): the hybrid's gain over BM25 is the one difference that is clear on both test sets. The wins/losses/ties columns show that every comparison is driven by a minority of queries, and
most queries are unaffected. An interval that includes 0 does not show that two systems are equal; it shows that this test set cannot separate
them.

**Code in the next cell.** `lab.timing_table()` now holds every timed step of the session, including Part B. It is saved to
`results/timings_all.csv`, and the total seconds per stage and collection are shown.

In [32]:
# Save the complete cost table of this session (setup / query / generate).
all_timings = lab.timing_table()
lab.save_table(all_timings, "timings_all.csv")
display(all_timings.groupby(["stage", "collection"], dropna=False)["seconds"].sum().round(2))

saved results/timings_all.csv


stage     collection
generate  scifact        3.23
query     nfcorpus       1.95
          scifact        9.03
setup     nfcorpus      58.20
          scifact       75.98
Name: seconds, dtype: float64